# NIH ChestX-ray14 — Classical ML Training (from extracted CSV)
### Loads `image_features_dataset.csv`, trains classical ML models, evaluates them, and saves the best one

**Sections 11-12** below build `image_features_dataset.csv` directly from the raw NIH metadata
table and the raw ChestX-ray14 image files — no external notebook is required. That CSV has one
row per image, with:

- `image_id`, `patient_id`, `disease_label` (raw pipe-separated string)
- `label_<Disease>` × 14 — multi-hot disease targets
- `feature_0000` … `feature_{FEATURE_DIM-1}` — frozen CNN embedding (e.g. 2048-d ResNet50 features)

**What this notebook does:**
0. **(Section 11-12)** Loads the raw NIH metadata, matches it to on-disk images, extracts a frozen
   ResNet50 feature vector per image, and writes the structured `image_features_dataset.csv`
1. Loads that CSV and reconstructs the feature matrix `X` and label matrix `y`
2. Performs a **patient-grouped** train/test split (same leakage-avoidance rule as the data pipeline)
3. Scales features and reduces dimensionality (`StandardScaler` + `PCA`) so classical models are tractable on high-dimensional CNN embeddings
4. Trains three classical ML models for **multi-label** disease classification:
   - Logistic Regression (One-vs-Rest)
   - Random Forest
   - Linear SVM (One-vs-Rest, with probability estimates)
5. Evaluates each model: **Accuracy, Precision, Recall, F1, ROC-AUC**
6. Generates **confusion matrices** — per-disease (multi-label) and an aggregate "Any Finding vs No Finding" view
7. Saves the best-performing trained model (plus its scaler/PCA) to disk with `joblib`

No knowledge graph construction happens in this notebook.

**Environment note:** all imports, third-party installs, and file-path configuration for the *entire* notebook (all pipelines below) are consolidated into the **Global Setup** section immediately below, so no later section re-imports a package or hardcodes a path.

## Environment Setup

**Objective:** make sure every third-party package this notebook depends on is installed before anything else runs.

**Implementation:** a single `pip install` for every package used anywhere below — the classical-ML stack (`scikit-learn`, `joblib`), the graph stack (`networkx`), and the deep-learning stack (`torch`, `torchvision`, `torchxrayvision`).

**Expected output:** a `✅ Environment dependencies installed` confirmation. Safe to re-run; `pip` skips packages that are already satisfied.

**Discussion:** Google Colab ships with `numpy`/`pandas`/`matplotlib`/`Pillow` preinstalled, so only the packages Colab doesn't already provide are listed explicitly.


In [ ]:
# Environment Setup — run once at the start of a fresh Colab/Jupyter session
!pip install -q torch torchvision torchxrayvision networkx scikit-learn joblib

print("✅ Environment dependencies installed (or already present)")


## Global Setup — Imports & Path Configuration

**Objective:** import every package this notebook uses, exactly once, and define every file path used later — so no section below re-imports a module or hardcodes an environment-specific path.

**Implementation:** one consolidated import block (covering the classical-ML, knowledge-graph, and deep-learning pipelines alike), followed by a `BASE_DIR`-rooted directory layout — `DATA_DIR` for inputs you upload, `KG_DIR` for intermediate graph/plot artifacts, `MODELS_DIR` for trained model bundles, and `OUTPUT_DIR` for final deliverables — created automatically with `os.makedirs`.

**Expected output:** a `✅` confirmation line per group, plus the resolved directory paths.

**Discussion:** `IN_COLAB` detection picks `/content/...` in Colab and a local subfolder otherwise, so the same notebook works unmodified in Colab, a local Jupyter install, or this validation environment. Every later section reuses `CSV_PATH`, `CSV_INPUT_PATH`, `IMAGE_DIR`, `METADATA_CSV`, `KG_DIR`, `MODELS_DIR`, `OUTPUT_DIR`, and `RANDOM_STATE` from this cell rather than redefining them.


In [ ]:
# ============================================================
# Global Imports — every third-party package used anywhere in this notebook,
# consolidated here once so no later section re-imports the same module.
# ============================================================
import os
import sys
import json
import glob
import shutil
import time
import pickle
import random
import itertools
import tracemalloc
from collections import Counter

import numpy as np
import pandas as pd
import joblib

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

import networkx as nx

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    multilabel_confusion_matrix,
    confusion_matrix,
    classification_report,
)

import torch
import torch.nn.functional as F
from PIL import Image

# Pretrained, chest-X-ray-specific deep learning models live in `torchxrayvision`.
# It ships DenseNet121 / ResNet checkpoints trained on real CXR corpora (including
# NIH ChestX-ray14), so its output head already speaks the right disease vocabulary —
# unlike a generic ImageNet-pretrained network, which would need a new classifier head
# and fine-tuning before its outputs meant anything clinically.
import torchvision
import torchvision.models as tv_models
import torchvision.transforms as T

try:
    import torchxrayvision as xrv
    HAVE_XRV = True
except ImportError:
    HAVE_XRV = False

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ Imports ready | torchxrayvision available: {HAVE_XRV} | device: {DEVICE}")

# ============================================================
# Path Configuration — every directory and input-file path used in this notebook
# ============================================================
IN_COLAB = "google.colab" in sys.modules
BASE_DIR = "/content/chestxray_kg_pipeline" if IN_COLAB else os.path.join(os.getcwd(), "chestxray_kg_pipeline")

DATA_DIR = os.path.join(BASE_DIR, "data")        # place/upload input files here
KG_DIR = os.path.join(BASE_DIR, "kg_workspace")  # intermediate graphs, pickles, PNGs
MODELS_DIR = os.path.join(BASE_DIR, "models")    # trained model bundles
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")   # final deliverables (GraphML, tables, figures)

for _dir in (BASE_DIR, DATA_DIR, KG_DIR, MODELS_DIR, OUTPUT_DIR):
    os.makedirs(_dir, exist_ok=True)

# Input file locations — edit these if your files live elsewhere (e.g. a mounted Google Drive)
CSV_PATH = os.path.join(DATA_DIR, "Data_Entry_2017_v2020.csv")         # NIH metadata table
CSV_INPUT_PATH = os.path.join(DATA_DIR, "image_features_dataset.csv")  # built by Sections 11-12 below
IMAGE_DIR = os.path.join(DATA_DIR, "images")                           # raw ChestX-ray14 PNG/JPEG files
METADATA_CSV = CSV_PATH                                                # same file, used for display labels only

print(f"✅ BASE_DIR:    {BASE_DIR}")
print(f"   DATA_DIR:    {DATA_DIR}  (place Data_Entry_2017_v2020.csv, image_features_dataset.csv, images/ here)")
print(f"   KG_DIR:      {KG_DIR}")
print(f"   MODELS_DIR:  {MODELS_DIR}")
print(f"   OUTPUT_DIR:  {OUTPUT_DIR}")


## Data Acquisition — mount Drive, download & extract ChestX-ray14 images, stage the CSV

**Objective:** get from "CSV + this notebook uploaded to the root of Google Drive" to "`CSV_PATH` and `IMAGE_DIR` both populated," with zero manual steps.

**Implementation:** mounts Google Drive, downloads the 12 official NIH `images_XX.tar.gz` archives into the Drive root (skipping any already downloaded), extracts each one into a flat `IMAGE_DIR` (flattening the archives' nested `images_NNN/images/` subfolders, since later sections glob `IMAGE_DIR` non-recursively), and copies `Data_Entry_2017_v2020.csv` from the Drive root into `DATA_DIR` so `CSV_PATH` resolves automatically. Everything is idempotent — safe to re-run or resume after a disconnect.

**Expected output:** `✅` confirmations for the drive mount, each downloaded archive, the extracted image count, and the staged CSV. Outside Colab (`IN_COLAB == False`) this cell is skipped — place files under `DATA_DIR` manually in that case.

**Discussion:** this replaces the standalone `batch_download_zips.py` script — folding it in here means the notebook runs top-to-bottom unmodified once the CSV and this notebook file are sitting in your Drive root.

In [ ]:
# Data Acquisition — download + extract ChestX-ray14 images, stage the CSV (Colab only)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    import tarfile

    # Root of your Google Drive — the CSV and (if you ran it standalone) the old
    # download script live directly here, no subfolder
    DRIVE_ROOT = '/content/drive/MyDrive'

    # --- 1) Download the 12 official NIH ChestX-ray14 archives into DRIVE_ROOT ---
    links = [
        'https://nihcc.box.com/shared/static/vfk49d74nhbxq3nqjg0900w5nvkorp5c.gz',
        'https://nihcc.box.com/shared/static/i28rlmbvmfjbl8p2n3ril0pptcmcu9d1.gz',
        'https://nihcc.box.com/shared/static/f1t00wrtdk94satdfb9olcolqx20z2jp.gz',
        'https://nihcc.box.com/shared/static/0aowwzs5lhjrceb3qp67ahp0rd1l1etg.gz',
        'https://nihcc.box.com/shared/static/v5e3goj22zr6h8tzualxfsqlqaygfbsn.gz',
        'https://nihcc.box.com/shared/static/asi7ikud9jwnkrnkj99jnpfkjdes7l6l.gz',
        'https://nihcc.box.com/shared/static/jn1b4mw4n6lnh74ovmcjb8y48h8xj07n.gz',
        'https://nihcc.box.com/shared/static/tvpxmn7qyrgl0w8wfh9kqfjskv6nmm1j.gz',
        'https://nihcc.box.com/shared/static/upyy3ml7qdumlgk2rfcvlb9k6gvqq2pj.gz',
        'https://nihcc.box.com/shared/static/l6nilvfa9cg3s28tqv1qc1olm3gnz54p.gz',
        'https://nihcc.box.com/shared/static/hhq8fkdgvcari67vfhs7ppg2w6ni4jze.gz',
        'https://nihcc.box.com/shared/static/ioqwiy20ihqwyr8pf4c24eazhh281pbu.gz',
    ]

    import urllib.request

    for idx, link in enumerate(links, start=1):
        fn = os.path.join(DRIVE_ROOT, 'images_%02d.tar.gz' % idx)
        if os.path.exists(fn):
            print(f'{os.path.basename(fn)} already exists, skipping')
            continue
        print(f'Downloading {os.path.basename(fn)} ...')
        try:
            urllib.request.urlretrieve(link, fn)
        except Exception as e:
            print(f'  \u26a0\ufe0f Failed to download {os.path.basename(fn)}: {e}')

    print('\u2705 Download step complete.')

    # --- 2) Extract each archive, flattened, into IMAGE_DIR ---
    EXTRACT_TMP = os.path.join(BASE_DIR, '_extract_tmp')
    os.makedirs(EXTRACT_TMP, exist_ok=True)

    archive_paths = sorted(glob.glob(os.path.join(DRIVE_ROOT, 'images_*.tar.gz')))
    assert len(archive_paths) > 0, f'No images_*.tar.gz files found under {DRIVE_ROOT}'

    for archive_path in archive_paths:
        archive_name = os.path.basename(archive_path)
        print(f'Extracting {archive_name} ...')

        with tarfile.open(archive_path, 'r:gz') as tar:
            tar.extractall(EXTRACT_TMP)

        moved = 0
        for img_path in glob.glob(os.path.join(EXTRACT_TMP, '**', '*.png'), recursive=True) + \
                         glob.glob(os.path.join(EXTRACT_TMP, '**', '*.jpg'), recursive=True):
            dest = os.path.join(IMAGE_DIR, os.path.basename(img_path))
            if not os.path.exists(dest):
                shutil.move(img_path, dest)
                moved += 1

        shutil.rmtree(EXTRACT_TMP)
        os.makedirs(EXTRACT_TMP, exist_ok=True)
        print(f'  -> moved {moved} images into {IMAGE_DIR}')

    shutil.rmtree(EXTRACT_TMP, ignore_errors=True)
    total_images = len(glob.glob(os.path.join(IMAGE_DIR, '*.png'))) + \
                    len(glob.glob(os.path.join(IMAGE_DIR, '*.jpg')))
    print(f'\u2705 Extraction complete: {total_images} images now flat under {IMAGE_DIR}')

    # --- 3) Stage the metadata CSV where CSV_PATH expects it ---
    csv_src = os.path.join(DRIVE_ROOT, 'Data_Entry_2017_v2020.csv')
    if os.path.exists(csv_src) and not os.path.exists(CSV_PATH):
        shutil.copy(csv_src, CSV_PATH)
        print(f'\u2705 Copied CSV to {CSV_PATH}')
    elif os.path.exists(CSV_PATH):
        print(f'CSV already present at {CSV_PATH}, skipping copy')
    else:
        print(f'\u26a0\ufe0f Could not find {csv_src} \u2014 upload Data_Entry_2017_v2020.csv to your Drive root')
else:
    print('Not running in Colab \u2014 skipping auto-download. '
          'Place Data_Entry_2017_v2020.csv and the image files under DATA_DIR manually.')

Mounted at /content/drive
images_01.tar.gz already exists, skipping
images_02.tar.gz already exists, skipping
images_03.tar.gz already exists, skipping
✅ Download step complete.

Extracting images_01.tar.gz
Finished images_01.tar.gz

Extracting images_02.tar.gz
Finished images_02.tar.gz

Extracting images_03.tar.gz
Finished images_03.tar.gz

Extracting images_04.tar.gz
Finished images_04.tar.gz

Extracting images_05.tar.gz
Finished images_05.tar.gz

Extracting images_06.tar.gz
Finished images_06.tar.gz

Extracting images_07.tar.gz
Finished images_07.tar.gz

Extracting images_08.tar.gz
Finished images_08.tar.gz

Extracting images_09.tar.gz
Finished images_09.tar.gz

Extracting images_10.tar.gz
Finished images_10.tar.gz

Extracting images_11.tar.gz
Finished images_11.tar.gz

Extracting images_12.tar.gz
Finished images_12.tar.gz

Finished extracting 112120 images.
✅ Copied CSV to /content/chestxray_kg_pipeline/data/Data_Entry_2017_v2020.csv


## Section 11 — Load raw ChestX-ray14 metadata & match it to on-disk images

**Objective:** load the NIH metadata table and determine exactly which rows have a corresponding
raw image file physically present under `IMAGE_DIR`, so feature extraction only runs on images that
actually exist.

**Implementation:** `pandas.read_csv` on `CSV_PATH`, the same column rename used later by the
structured knowledge-graph section, plus a set-intersection against the filenames found under
`IMAGE_DIR`. The full 14-disease vocabulary (`DISEASE_LIST`) is derived from the metadata's
pipe-separated `finding_labels` column rather than hardcoded.

**Expected output:** `raw_meta` (every metadata row), `meta_matched` (only the rows with an on-disk
image), and `DISEASE_LIST` (the 14 ChestX-ray14 disease names, excluding "No Finding").

**Discussion:** this mirrors the loading logic used later in the structured Knowledge Graph section,
kept as a separate, independent pass here since this pipeline needs the disease vocabulary and the
matched-image list *before* graph construction runs.


In [ ]:
# D1 — Load raw NIH metadata and match rows to image files physically present in IMAGE_DIR
assert os.path.exists(CSV_PATH), (
    f"Could not find {CSV_PATH}. Upload Data_Entry_2017_v2020.csv into {DATA_DIR} "
    f"(or point CSV_PATH at its location in Global Setup) before running this section."
)

raw_meta = pd.read_csv(CSV_PATH)
raw_meta.columns = [
    "image_index", "finding_labels", "follow_up", "patient_id",
    "patient_age", "patient_sex", "view_position",
    "orig_width", "orig_height", "pixel_spacing_x", "pixel_spacing_y",
]
raw_meta["finding_list"] = raw_meta["finding_labels"].apply(lambda s: s.split("|"))

DISEASE_LIST = sorted({f for findings in raw_meta["finding_list"] for f in findings if f != "No Finding"})

available_images = {
    os.path.basename(p) for p in
    glob.glob(os.path.join(IMAGE_DIR, "*.png")) + glob.glob(os.path.join(IMAGE_DIR, "*.jpg"))
}
assert len(available_images) > 0, (
    f"No images found under {IMAGE_DIR}. Place the raw ChestX-ray14 PNG/JPEG files there before "
    f"running the feature-extraction pipeline."
)

meta_matched = raw_meta[raw_meta["image_index"].isin(available_images)].reset_index(drop=True)
assert len(meta_matched) > 0, (
    f"None of the {len(raw_meta):,} metadata rows in {CSV_PATH} match a filename under {IMAGE_DIR}."
)

print(f"\u2705 Loaded metadata for {len(raw_meta):,} rows ({len(DISEASE_LIST)} disease labels: {DISEASE_LIST})")
print(f"\u2705 Found {len(available_images):,} raw image files under {IMAGE_DIR}")
print(f"\u2705 Matched {len(meta_matched):,} metadata rows to on-disk images (feature extraction runs on these)")


Streaming output truncated to the last 5000 lines.
00006087_000.png   00012904_008.png   00019409_000.png	 00028897_017.png
00006088_000.png   00012905_000.png   00019410_000.png	 00028897_018.png
00006089_000.png   00012905_001.png   00019411_000.png	 00028897_019.png
00006090_000.png   00012905_002.png   00019412_000.png	 00028897_020.png
00006090_001.png   00012905_003.png   00019413_000.png	 00028898_000.png
00006091_000.png   00012905_004.png   00019413_001.png	 00028899_000.png
00006091_001.png   00012905_005.png   00019414_000.png	 00028899_001.png
00006092_000.png   00012905_006.png   00019415_000.png	 00028899_002.png
00006093_000.png   00012906_000.png   00019415_001.png	 00028900_000.png
00006094_000.png   00012907_000.png   00019415_002.png	 00028901_000.png
00006095_000.png   00012907_001.png   00019416_000.png	 00028901_001.png
00006095_001.png   00012907_002.png   00019417_000.png	 00028902_000.png
00006096_000.png   00012907_003.png   00019417_001.png	 00028902_001.png


## Section 12 — Extract CNN image features and export the structured CSV dataset

**Objective:** turn each raw, unstructured X-ray image into a fixed-length numeric feature vector,
and assemble the resulting structured dataset (`image_features_dataset.csv`) that Section 14 onward,
and the structured Knowledge Graph, both build on.

**Implementation:** a frozen, ImageNet-pretrained `ResNet50` (classifier head removed) is used as a
fixed feature extractor — no fine-tuning, no gradient updates — producing one 2048-d pooled feature
vector per image. Images are read from `IMAGE_DIR`, converted to 3-channel RGB, resized to 224x224,
and ImageNet-normalized, processed in batches for efficiency. Multi-hot disease labels are built from
`finding_list` against the shared `DISEASE_LIST`. The final table — `image_id`, `patient_id`,
`disease_label`, one `label_<Disease>` column per disease, and `feature_0000` … `feature_2047` — is
written to `CSV_INPUT_PATH`.

**Expected output:** a `✅ Saved structured feature dataset` confirmation with the final shape, plus
`image_features_dataset.csv` on disk under `DATA_DIR`.

**Discussion:** using a generic, frozen ImageNet backbone (rather than a CXR-specific one) is a
deliberate contrast with the *second*, direct multimedia pipeline later in this notebook, which uses
a domain-specific pretrained CXR model (`torchxrayvision`) — the classical-ML pipeline only needs a
generic, fixed-length embedding, while the direct pipeline needs disease-aware outputs. Running this
cell can take a while on the full dataset; it prints incremental progress and only needs to be run
once per session, since its output is cached to disk.


In [ ]:
# D2 — Frozen, ImageNet-pretrained ResNet50 as a fixed feature extractor (backbone only, no classifier head)
FEATURE_DIM = 2048

feature_extractor = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
feature_extractor.fc = torch.nn.Identity()   # drop the 1000-way ImageNet head -> raw 2048-d pooled features
feature_extractor.eval().to(DEVICE)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
resnet_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def extract_resnet_features(image_filenames, batch_size=32):
    """Reads each image from IMAGE_DIR, converts grayscale CXR to 3-channel RGB, and returns
    an [N, FEATURE_DIM] numpy array of frozen ResNet50 pooled features (no fine-tuning)."""
    feats = np.zeros((len(image_filenames), FEATURE_DIM), dtype=np.float32)
    with torch.no_grad():
        for start in range(0, len(image_filenames), batch_size):
            batch_names = image_filenames[start:start + batch_size]
            batch_tensors = [
                resnet_transform(Image.open(os.path.join(IMAGE_DIR, fname)).convert("RGB"))
                for fname in batch_names
            ]
            batch = torch.stack(batch_tensors).to(DEVICE)
            out = feature_extractor(batch).cpu().numpy()
            feats[start:start + len(batch_names)] = out
            done = min(start + batch_size, len(image_filenames))
            print(f"\r   Extracted features for {done:,}/{len(image_filenames):,} images", end="")
    print()
    return feats


t_extract0 = time.time()
image_filenames = meta_matched["image_index"].tolist()
feature_matrix = extract_resnet_features(image_filenames)
print(f"\u2705 Extracted {feature_matrix.shape[1]}-d ResNet50 features for {feature_matrix.shape[0]:,} images "
      f"in {time.time() - t_extract0:.1f}s")


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 223MB/s]


   Extracted features for 18,016/112,120 images

In [ ]:
# D3 — Multi-hot disease labels + assembled structured CSV (image_features_dataset.csv)
label_matrix = np.zeros((len(meta_matched), len(DISEASE_LIST)), dtype=np.int32)
for i, findings in enumerate(meta_matched["finding_list"]):
    for f in findings:
        if f in DISEASE_LIST:
            label_matrix[i, DISEASE_LIST.index(f)] = 1

feature_df = pd.DataFrame(feature_matrix, columns=[f"feature_{i:04d}" for i in range(FEATURE_DIM)])
label_df = pd.DataFrame(label_matrix, columns=[f"label_{d}" for d in DISEASE_LIST])

structured_dataset_df = pd.concat([
    meta_matched[["image_index", "patient_id", "finding_labels"]].rename(
        columns={"image_index": "image_id", "finding_labels": "disease_label"}
    ),
    label_df,
    feature_df,
], axis=1)

structured_dataset_df.to_csv(CSV_INPUT_PATH, index=False)

print(f"\u2705 Saved structured feature dataset -> {CSV_INPUT_PATH}")
print(f"   Shape: {structured_dataset_df.shape[0]:,} rows x {structured_dataset_df.shape[1]:,} columns")
print(f"   ({len(DISEASE_LIST)} disease labels, {FEATURE_DIM} CNN feature dims)")


## Section 13 — Imports

**Objective:** confirm the packages this pipeline needs are available.

**Implementation:** all imports for this entire notebook — this pipeline included — are installed and imported once in the **Global Setup** section above, so this section has no code cell of its own; it previously duplicated `numpy`/`pandas`/`sklearn` imports already established globally.

**Expected output:** none — this is a pointer back to Global Setup.

**Discussion:** consolidating imports avoids the maintenance risk of two import lists silently drifting apart as the notebook evolves.

## Section 14 — Load the generated CSV

Reads `image_features_dataset.csv` (produced by Sections 11-12 above) and reconstructs:

- `X` — the `[N, FEATURE_DIM]` CNN feature matrix, from the `feature_*` columns
- `y` — the `[N, 14]` multi-hot disease label matrix, from the `label_*` columns
- `groups` — `patient_id`, used for a leakage-free split in Section 15

**Code Explanation:** `feature_cols`/`label_cols` are recovered from the CSV's column prefixes rather than hardcoded, so this cell tolerates a different `FEATURE_DIM` or disease list without edits.

**Expected output:** `X`, `y`, and `groups` arrays plus a per-disease positive-rate summary.

**Discussion:** if `CSV_INPUT_PATH` doesn't exist, the assertion below fails with a clear message rather than a cryptic `pandas` error — run Sections 11-12 above first to generate it.

In [ ]:
# R14 — Load the CSV produced by the data pipeline notebook
# CSV_INPUT_PATH is defined once in Global Setup (edit it there, not here)
assert os.path.exists(CSV_INPUT_PATH), (
    f"Could not find {CSV_INPUT_PATH}. Run Sections 11-12 above first "
    f"to generate it, or point CSV_INPUT_PATH at its location."
)

df = pd.read_csv(CSV_INPUT_PATH)

label_cols = [c for c in df.columns if c.startswith("label_")]
feature_cols = [c for c in df.columns if c.startswith("feature_")]
DISEASE_NAMES = [c.replace("label_", "") for c in label_cols]

assert len(label_cols) > 0, "No label_* columns found — check the CSV."
assert len(feature_cols) > 0, "No feature_* columns found — check the CSV."

X = df[feature_cols].to_numpy(dtype=np.float32)
y = df[label_cols].to_numpy(dtype=np.int32)
groups = df["patient_id"].to_numpy()

print(f"✅ Loaded {CSV_INPUT_PATH}: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"   X shape: {X.shape}")
print(f"   y shape: {y.shape}  ({len(DISEASE_NAMES)} diseases: {DISEASE_NAMES})")
print(f"   Unique patients: {len(np.unique(groups))}")
print(f"   Positive rate per disease (top 5):")
print(pd.Series(y.sum(axis=0), index=DISEASE_NAMES).sort_values(ascending=False).head())

## Section 15 — Patient-grouped train/test split

Same rule as Sections 11-12 above: images from the same patient must never appear in both
the train and test sets, or the model could "cheat" by recognizing a patient's anatomy rather than
learning disease features.

**Objective:** split into train/test without leaking a patient across the boundary.

**Implementation:** `GroupShuffleSplit` grouped on `patient_id`, with an explicit overlap assertion.

**Expected output:** train/test sizes plus a `✅ No patient overlap` confirmation.

**Discussion:** the assertion turns a subtle leakage bug into a loud, immediate failure.

In [ ]:
# R15 — Patient-grouped train/test split (80/20)
TEST_SIZE = 0.2
RANDOM_STATE = 42

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

overlap = set(groups_train) & set(groups_test)
assert len(overlap) == 0, f"Patient leakage detected: {len(overlap)} patients in both splits!"

print(f"✅ Train: {X_train.shape[0]} images from {len(set(groups_train))} patients")
print(f"✅ Test:  {X_test.shape[0]} images from {len(set(groups_test))} patients")
print(f"✅ No patient overlap between splits")

## Section 16 — Feature scaling + dimensionality reduction

CNN embeddings are high-dimensional (e.g. 2048-d for ResNet50). Classical ML models — especially
SVM — train far more efficiently, and often generalize better, on a lower-dimensional, standardized
representation. `StandardScaler` is fit on the training split only, then `PCA` compresses the scaled
features while retaining most of their variance. Both transforms are fit on **train only** and just
applied (`.transform`) to test, to avoid leaking test information into the model.

**Objective:** make high-dimensional CNN embeddings tractable for classical ML.

**Implementation:** `StandardScaler` then `PCA`, both fit on train only.

**Expected output:** the reduced dimensionality and cumulative explained variance.

**Discussion:** fitting only on train (never `X_test`) avoids leaking test-set structure into the transform.

In [ ]:
# R16 — Standardize features, then reduce dimensionality with PCA
N_PCA_COMPONENTS = min(100, X_train.shape[0] - 1, X_train.shape[1])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

pca = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_STATE)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

explained = pca.explained_variance_ratio_.sum()
print(f"✅ PCA: {X_train.shape[1]} -> {N_PCA_COMPONENTS} dims")
print(f"   Cumulative explained variance: {explained:.3f}")

## Section 17 — Define and train classical ML models

Three classical multi-label classifiers, each wrapped in `OneVsRestClassifier` so that one binary
classifier is trained per disease, and each exposes `predict_proba` for ROC-AUC:

- **Logistic Regression** — fast linear baseline
- **Random Forest** — non-linear, handles feature interactions, gives feature importances
- **Linear SVM** (`SVC(kernel="linear", probability=True)`) — strong linear margin classifier

**Objective:** train three complementary multi-label classifiers.

**Implementation:** each wrapped in `OneVsRestClassifier` with `class_weight="balanced"` to offset the dataset's disease-prevalence imbalance.

**Expected output:** three fitted models in `trained_models`.

**Discussion:** balancing class weight matters here — most diseases are rare positives against a large "No Finding" majority.

In [ ]:
# R17 — Define models
MODELS = {
    "Logistic Regression": OneVsRestClassifier(
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
    ),
    "Random Forest": OneVsRestClassifier(
        RandomForestClassifier(
            n_estimators=300, max_depth=None, class_weight="balanced",
            n_jobs=-1, random_state=RANDOM_STATE,
        )
    ),
    "Linear SVM": OneVsRestClassifier(
        SVC(kernel="linear", probability=True, class_weight="balanced", random_state=RANDOM_STATE)
    ),
}

trained_models = {}
for name, clf in MODELS.items():
    print(f"Training {name}...")
    clf.fit(X_train_pca, y_train)
    trained_models[name] = clf
    print(f"  ✅ {name} trained")

print("\n✅ All models trained")

## Section 18 — Evaluate: Accuracy, Precision, Recall, F1, ROC-AUC

Because this is a **multi-label** problem (each image can have 0, 1, or several diseases), metrics
are computed two ways for context:

- **Subset accuracy** (`accuracy_score`) — the strict metric: an image only counts as "correct" if
  *every* disease label matches exactly.
- **Macro-averaged** Precision / Recall / F1 / ROC-AUC — computed per disease, then averaged, so rare
  diseases count as much as common ones.

`predict_proba` on a `OneVsRestClassifier` returns one probability column per disease, which is what
ROC-AUC needs.

**Code Explanation:** subset accuracy is strict (every label must match); macro metrics average per-disease so rare diseases count equally with common ones; ROC-AUC only considers diseases with both classes present in `y_test`.

**Expected output:** `results_df`, one row per model, rounded to 4 decimals.

**Discussion:** reporting both a strict and a macro-averaged view avoids the model looking artificially strong (or weak) from a single metric choice.

In [ ]:
# R18 — Evaluate each model
results = []
predictions = {}   # name -> (y_pred, y_score) for use in Section 19

for name, clf in trained_models.items():
    y_pred = clf.predict(X_test_pca)
    y_score = clf.predict_proba(X_test_pca)
    predictions[name] = (y_pred, y_score)

    acc = accuracy_score(y_test, y_pred)                      # strict subset accuracy
    precision = precision_score(y_test, y_pred, average="macro", zero_division=0)
    recall = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

    # ROC-AUC only defined for diseases with both classes present in y_test
    valid_cols = [i for i in range(y_test.shape[1]) if len(np.unique(y_test[:, i])) > 1]
    if valid_cols:
        roc_auc = roc_auc_score(y_test[:, valid_cols], y_score[:, valid_cols], average="macro")
    else:
        roc_auc = float("nan")

    results.append({
        "Model": name,
        "Accuracy (subset)": acc,
        "Precision (macro)": precision,
        "Recall (macro)": recall,
        "F1 (macro)": f1,
        "ROC-AUC (macro)": roc_auc,
    })

results_df = pd.DataFrame(results).set_index("Model")
print("✅ Evaluation complete\n")
results_df.round(4)

In [ ]:
# R18b — Per-disease classification report for the best model (by F1), for a closer look
best_model_name = results_df["F1 (macro)"].idxmax()
print(f"Best model by macro F1: {best_model_name}\n")

y_pred_best, y_score_best = predictions[best_model_name]
print(classification_report(
    y_test, y_pred_best, target_names=DISEASE_NAMES, zero_division=0
))

## Section 19 — Confusion matrices

Two complementary views:

1. **Per-disease confusion matrices** (`multilabel_confusion_matrix`) — one 2×2 matrix per disease
   for the best model, shown as a grid of heatmaps.
2. **Aggregate "Any Finding vs No Finding" confusion matrix** — collapses all 14 diseases into a
   single binary target (does the image have *any* finding at all?) for one easy-to-read overall
   confusion matrix.

**Objective:** see *where* each model's errors are concentrated, not just aggregate scores.

**Implementation:** per-disease multilabel confusion matrices for the best model, plus a collapsed "Any Finding vs No Finding" binary view.

**Expected output:** a grid of 2x2 heatmaps, plus one aggregate heatmap.

**Discussion:** the aggregate view is the easiest one to sanity-check against `Accuracy (subset)` in Section 18.

In [ ]:
# R19a — Per-disease confusion matrices (best model)
mcm = multilabel_confusion_matrix(y_test, y_pred_best)

n_diseases = len(DISEASE_NAMES)
n_cols = 5
n_rows = int(np.ceil(n_diseases / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2 * n_cols, 3 * n_rows))
axes = axes.flatten()

for i, name in enumerate(DISEASE_NAMES):
    ax = axes[i]
    cm = mcm[i]
    ax.imshow(cm, cmap="Blues")
    for r in range(2):
        for c in range(2):
            ax.text(c, r, str(cm[r, c]), ha="center", va="center", fontsize=10)
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(["Pred 0", "Pred 1"], fontsize=7)
    ax.set_yticklabels(["True 0", "True 1"], fontsize=7)
    ax.set_title(name, fontsize=9)

for j in range(n_diseases, len(axes)):
    axes[j].axis("off")

plt.suptitle(f"Per-disease confusion matrices — {best_model_name}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# R19b — Aggregate binary confusion matrix: Any Finding vs No Finding
y_test_any = (y_test.sum(axis=1) > 0).astype(int)
y_pred_any = (y_pred_best.sum(axis=1) > 0).astype(int)

cm_any = confusion_matrix(y_test_any, y_pred_any)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm_any, cmap="Blues")
for r in range(2):
    for c in range(2):
        ax.text(c, r, str(cm_any[r, c]), ha="center", va="center", fontsize=14)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["No Finding", "Any Finding"])
ax.set_yticklabels(["No Finding", "Any Finding"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title(f"Any Finding vs No Finding — {best_model_name}")
plt.tight_layout()
plt.show()

print(f"Accuracy (Any Finding vs No Finding): {accuracy_score(y_test_any, y_pred_any):.4f}")

## Section 20 — Model comparison summary

**Objective:** give a single, at-a-glance visual comparison of the three trained models across every metric computed in Section 18.

**Implementation:** the `results_df` table already built in Section 18 is plotted directly as a grouped bar chart — no re-computation, just a different view of the same numbers.

**Expected output:** one bar chart with Accuracy/Precision/Recall/F1/ROC-AUC grouped per model, plus the underlying table for reference.

**Discussion:** a visual summary makes trade-offs (e.g. one model with higher recall but lower precision) easier to spot than scanning a table of numbers.

In [ ]:
# R20 — Bar chart comparing all models across metrics
metrics_to_plot = ["Accuracy (subset)", "Precision (macro)", "Recall (macro)", "F1 (macro)", "ROC-AUC (macro)"]

ax = results_df[metrics_to_plot].plot(kind="bar", figsize=(10, 5), rot=0)
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Classical ML model comparison — ChestX-ray14 multi-label classification")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

results_df.round(4)

## Section 21 — Save the trained model

Saves the best-performing model (by macro F1), bundled together with the `StandardScaler` and `PCA`
transforms it depends on, plus the disease name ordering — everything needed to run inference on new
raw feature vectors later without retraining.

**Objective:** persist the winning model so inference doesn't require retraining.

**Implementation:** bundle the model with its `scaler`/`pca`/disease-name ordering — everything downstream inference needs — via `joblib.dump`.

**Expected output:** a `.joblib` file under `MODELS_DIR` and a results CSV under `OUTPUT_DIR`.

**Discussion:** bundling the preprocessing steps with the model avoids a common inference bug — applying a *new* `StandardScaler`/`PCA` instead of the ones the model was trained on.

In [ ]:
# R21 — Save the best model (+ its preprocessing pipeline) to disk
MODEL_OUTPUT_PATH = os.path.join(
    MODELS_DIR, f"chestxray14_{best_model_name.lower().replace(' ', '_')}.joblib"
)

bundle = {
    "model_name": best_model_name,
    "model": trained_models[best_model_name],
    "scaler": scaler,
    "pca": pca,
    "disease_names": DISEASE_NAMES,
    "feature_cols": feature_cols,
    "metrics": results_df.loc[best_model_name].to_dict(),
}

joblib.dump(bundle, MODEL_OUTPUT_PATH)
print(f"✅ Saved best model ({best_model_name}) to {MODEL_OUTPUT_PATH}")

# Also save the full results table alongside the model for reference
RESULTS_OUTPUT_PATH = os.path.join(OUTPUT_DIR, "model_evaluation_results.csv")
results_df.to_csv(RESULTS_OUTPUT_PATH)
print(f"✅ Saved evaluation results to {RESULTS_OUTPUT_PATH}")

### Loading the saved model later

```python
import joblib
bundle = joblib.load(os.path.join(MODELS_DIR, "chestxray14_<model_name>.joblib"))

model = bundle["model"]
scaler = bundle["scaler"]
pca = bundle["pca"]
disease_names = bundle["disease_names"]

# For a new row of raw feature_* values (shape [1, FEATURE_DIM]):
X_new_scaled = scaler.transform(X_new)
X_new_pca = pca.transform(X_new_scaled)
y_pred = model.predict(X_new_pca)          # multi-hot prediction
y_score = model.predict_proba(X_new_pca)   # per-disease probabilities
```

## Notes

- No knowledge graph is built in this notebook — it stops at trained, evaluated, saved classical ML
  models, as requested.
- `N_PCA_COMPONENTS`, `TEST_SIZE`, and the model hyperparameters in Section 16–17 are reasonable
  defaults; tune them once real extracted features are available (e.g. increase PCA components if
  explained variance is low, or add `GridSearchCV` for hyperparameter search).


**Objective:** construct the structured (metadata-only) knowledge graph.

**Implementation:** load `Data_Entry_2017_v2020.csv`, then build a `networkx.MultiDiGraph` whose nodes and edges are read directly off its columns — no images, no ML features.

# ChestX-ray14 Knowledge Graph Construction

Builds a knowledge graph exclusively from `Data_Entry_2017_v2020.csv`
(the NIH ChestX-ray14 metadata table).

## Ontology

**Node types (4):**
1. `Patient` — id: patient_id
2. `Image` — id: image_index (filename)
3. `Finding` — id: finding name (14 diseases + "No Finding")
4. `ViewPosition` — id: PA | AP

**Relationship types (4):**
1. `(Patient) -[:HAS_IMAGE]-> (Image)` — per-edge attr: follow_up_number
2. `(Image) -[:HAS_FINDING]-> (Finding)`
3. `(Image) -[:TAKEN_IN_VIEW]-> (ViewPosition)`
4. `(Finding) -[:CO_OCCURS_WITH]-> (Finding)` — derived, symmetric, attr: weight (co-occurrence count)

No image pixel data, no ML features, no external knowledge is used —
every node/edge is derived strictly from columns in the CSV.
**Expected output:** an in-memory `MultiDiGraph` `G`, persisted to `KG_DIR` for reuse by later sections.

**Discussion:** because every node/edge maps to a spreadsheet cell, this graph is fully auditable — a property revisited in the comparison sections later in the notebook.

In [ ]:
# CSV_PATH is defined once in Global Setup; reused here unchanged
t0 = time.time()


## 1. Load & normalize CSV

**Objective:** load the NIH metadata table and give its columns clear, consistent names.

**Implementation:** `pandas.read_csv` on `CSV_PATH`, followed by an explicit column rename (the raw NIH file ships verbose, space-containing column names) and a pipe-split of the multi-label `finding_labels` string into `finding_list`.

**Expected output:** `df`, with a row count printed for a quick sanity check.

**Discussion:** normalizing column names here means every downstream cell can rely on the same short, snake_case names instead of re-deriving them.

In [ ]:
df = pd.read_csv(CSV_PATH)
df.columns = [
    "image_index", "finding_labels", "follow_up", "patient_id",
    "patient_age", "patient_sex", "view_position",
    "orig_width", "orig_height", "pixel_spacing_x", "pixel_spacing_y",
]
df["finding_list"] = df["finding_labels"].apply(lambda s: s.split("|"))

print(f"Loaded {len(df):,} rows from {CSV_PATH}")

## 2. Build the graph

**Objective:** turn the tabular metadata into a typed, multi-relational graph.

**Implementation:** `Finding` and `ViewPosition` nodes are created up front from the CSV's distinct values; `Patient` and `Image` nodes, plus the `HAS_IMAGE` / `HAS_FINDING` / `TAKEN_IN_VIEW` edges, are built incrementally in one pass over `df` (next cell); the derived, symmetric `CO_OCCURS_WITH` edges are added once co-occurrence counts are finalized.

**Expected output:** the node/edge scaffolding for `G`, completed by the following two cells.

**Discussion:** building `Patient` attributes as running aggregates (rather than one row per visit) avoids duplicate `Patient` nodes for patients with multiple imaging visits.

In [ ]:
G = nx.MultiDiGraph(name="ChestXray14_KnowledgeGraph")

ALL_FINDINGS = sorted(set(itertools.chain.from_iterable(df["finding_list"])))
VIEW_POSITIONS = sorted(df["view_position"].unique())

# --- Finding nodes ---
for f in ALL_FINDINGS:
    G.add_node(
        f"Finding::{f}",
        node_type="Finding",
        name=f,
        is_no_finding=(f == "No Finding"),
    )

# --- ViewPosition nodes ---
for v in VIEW_POSITIONS:
    G.add_node(
        f"View::{v}",
        node_type="ViewPosition",
        name=v,
    )

In [ ]:
# --- Patient node aggregates (built incrementally) ---
patient_seen = {}  # patient_id -> dict of running stats

finding_pair_counts = {}  # (f1,f2) -> co-occurrence weight

for row in df.itertuples(index=False):
    pid = f"Patient::{row.patient_id}"
    img_id = f"Image::{row.image_index}"

    # --- Patient node (create or update aggregate attrs) ---
    if pid not in patient_seen:
        patient_seen[pid] = {
            "sexes": set(),
            "ages": [],
            "n_images": 0,
        }
        G.add_node(pid, node_type="Patient", patient_id=int(row.patient_id))
    stats = patient_seen[pid]
    stats["sexes"].add(row.patient_sex)
    stats["ages"].append(row.patient_age)
    stats["n_images"] += 1

    # --- Image node ---
    G.add_node(
        img_id,
        node_type="Image",
        image_index=row.image_index,
        follow_up_number=int(row.follow_up),
        patient_age_at_capture=int(row.patient_age),
        patient_sex=row.patient_sex,
        orig_width=int(row.orig_width),
        orig_height=int(row.orig_height),
        pixel_spacing_x=float(row.pixel_spacing_x),
        pixel_spacing_y=float(row.pixel_spacing_y),
    )

    # --- Relationship: Patient -HAS_IMAGE-> Image ---
    G.add_edge(pid, img_id, key="HAS_IMAGE", relation="HAS_IMAGE",
               follow_up_number=int(row.follow_up))

    # --- Relationship: Image -TAKEN_IN_VIEW-> ViewPosition ---
    G.add_edge(img_id, f"View::{row.view_position}", key="TAKEN_IN_VIEW",
               relation="TAKEN_IN_VIEW")

    # --- Relationship: Image -HAS_FINDING-> Finding ---
    findings = row.finding_list
    for f in findings:
        G.add_edge(img_id, f"Finding::{f}", key=f"HAS_FINDING::{f}",
                   relation="HAS_FINDING")

    # --- Track Finding co-occurrence (excluding "No Finding") ---
    real_findings = [f for f in findings if f != "No Finding"]
    for f1, f2 in itertools.combinations(sorted(real_findings), 2):
        finding_pair_counts[(f1, f2)] = finding_pair_counts.get((f1, f2), 0) + 1

In [ ]:
# --- Finalize Patient node attributes (aggregate) ---
for pid, stats in patient_seen.items():
    G.nodes[pid]["sex"] = "/".join(sorted(stats["sexes"]))  # usually single value
    G.nodes[pid]["age_min"] = min(stats["ages"])
    G.nodes[pid]["age_max"] = max(stats["ages"])
    G.nodes[pid]["n_images"] = stats["n_images"]

# --- Relationship: Finding -CO_OCCURS_WITH-> Finding (derived, symmetric) ---
for (f1, f2), w in finding_pair_counts.items():
    G.add_edge(f"Finding::{f1}", f"Finding::{f2}", key="CO_OCCURS_WITH",
               relation="CO_OCCURS_WITH", weight=w)
    G.add_edge(f"Finding::{f2}", f"Finding::{f1}", key="CO_OCCURS_WITH",
               relation="CO_OCCURS_WITH", weight=w)

print(f"Graph built in {time.time()-t0:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}   Edges: {G.number_of_edges():,}")

## 3. Summary stats by type

**Objective:** sanity-check the graph immediately after construction.

**Implementation:** `Counter` over each node's `node_type` and each edge's `relation` attribute.

**Expected output:** a printed breakdown of node and relationship counts by type.

**Discussion:** these counts are the first thing to check if a later section's node/edge counts look wrong — they establish the ground truth for `G` right after it's built.

In [ ]:
node_type_counts = Counter(nx.get_node_attributes(G, "node_type").values())
edge_type_counts = Counter(d["relation"] for _, _, d in G.edges(data=True))

print("\nNode types:")
for k, v in node_type_counts.items():
    print(f"  {k:15s} {v:,}")

print("\nRelationship types:")
for k, v in edge_type_counts.items():
    print(f"  {k:20s} {v:,}")

## 4. Persist graph object + stats for downstream steps

**Objective:** make `G` and its supporting artifacts durable, so later sections (or a fresh kernel) can reload them without recomputation.

**Implementation:** pickle `G` and `finding_pair_counts` and save `df` under `KG_DIR`.

**Expected output:** `graph.gpickle`, `finding_pair_counts.pkl`, and `df.pkl` written to `KG_DIR`.

**Discussion:** `G` also stays live in memory for the rest of this notebook session — the pickle files exist so any individual section can be re-run independently later without re-running everything above it.

In [ ]:
with open(os.path.join(KG_DIR, "graph.gpickle"), "wb") as fh:
    pickle.dump(G, fh)

with open(os.path.join(KG_DIR, "finding_pair_counts.pkl"), "wb") as fh:
    pickle.dump(finding_pair_counts, fh)

df.to_pickle(os.path.join(KG_DIR, "df.pkl"))

print("\nSaved graph.gpickle, finding_pair_counts.pkl, df.pkl")

# Visualize the ChestX-ray14 Knowledge Graph

**Objective:** visually inspect `G` at three levels of granularity — schema, a sample neighborhood, and the finding co-occurrence network — to confirm the graph looks like the ontology it was designed to encode.

**Implementation:** reloads `G` from `KG_DIR` (so this section can run standalone), then builds three matplotlib figures.

**Expected output:** three PNGs saved under `KG_DIR` and displayed inline.

**Discussion:** visual inspection catches ontology bugs (e.g. a relationship pointing the wrong direction) that pass silently through the numeric checks in Section 3.

In [ ]:
# Reload G (lets this visualization section run standalone from a fresh kernel)
random.seed(42)

with open(os.path.join(KG_DIR, "graph.gpickle"), "rb") as fh:
    G = pickle.load(fh)

COLORS = {
    "Patient": "#4C72B0",
    "Image": "#55A868",
    "Finding": "#C44E52",
    "ViewPosition": "#8172B2",
}

## 1. Ontology / Schema Diagram

**Objective:** show the meta-graph (node types -> relationship types) rather than the full, unreadable node-level graph.

**Implementation:** a small hand-laid-out schema graph mirroring the ontology from Section 24.

**Expected output:** `01_ontology_schema.png` — four node-type boxes connected by labeled relationship arrows.

**Discussion:** this is the fastest way to confirm the ontology matches what was actually built, before drilling into real data below.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

S = nx.MultiDiGraph()
for nt in ["Patient", "Image", "Finding", "ViewPosition"]:
    S.add_node(nt)
S.add_edge("Patient", "Image", label="HAS_IMAGE")
S.add_edge("Image", "Finding", label="HAS_FINDING")
S.add_edge("Image", "ViewPosition", label="TAKEN_IN_VIEW")
S.add_edge("Finding", "Finding", label="CO_OCCURS_WITH")

pos = {
    "Patient": (0, 0),
    "Image": (3, 0),
    "Finding": (6, 1.2),
    "ViewPosition": (6, -1.2),
}

for nt, (x, y) in pos.items():
    ax.add_patch(mpatches.FancyBboxPatch(
        (x - 0.9, y - 0.4), 1.8, 0.8,
        boxstyle="round,pad=0.05,rounding_size=0.1",
        linewidth=2, edgecolor=COLORS[nt], facecolor=COLORS[nt] + "33"))
    ax.text(x, y, nt, ha="center", va="center", fontsize=13, fontweight="bold", color=COLORS[nt])

def draw_edge(a, b, label, curve=0.0, self_loop=False):
    xa, ya = pos[a]
    xb, yb = pos[b]
    if self_loop:
        ax.annotate("", xy=(xb + 0.9, yb + 0.5), xytext=(xb + 0.9, yb - 0.5),
                    arrowprops=dict(arrowstyle="-|>", color="#555555", lw=1.8,
                                     connectionstyle="arc3,rad=1.4"))
        ax.text(xb + 2.0, yb, label, fontsize=9.5, ha="left", va="center", style="italic")
    else:
        ax.annotate("", xy=(xb - 0.9, yb), xytext=(xa + 0.9, ya),
                    arrowprops=dict(arrowstyle="-|>", color="#555555", lw=1.8,
                                     connectionstyle=f"arc3,rad={curve}"))
        mx, my = (xa + xb) / 2, (ya + yb) / 2 + curve * 1.5
        ax.text(mx, my + 0.25, label, fontsize=9.5, ha="center", va="center", style="italic")

draw_edge("Patient", "Image", "HAS_IMAGE")
draw_edge("Image", "Finding", "HAS_FINDING", curve=0.15)
draw_edge("Image", "ViewPosition", "TAKEN_IN_VIEW", curve=-0.15)
draw_edge("Finding", "Finding", "CO_OCCURS_WITH", self_loop=True)

ax.set_xlim(-2, 9.5)
ax.set_ylim(-2.5, 2.5)
ax.axis("off")
ax.set_title("ChestX-ray14 Knowledge Graph — Ontology (Schema)", fontsize=15, fontweight="bold", pad=20)
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "01_ontology_schema.png"), dpi=180, bbox_inches="tight")
plt.close()
print("Saved 01_ontology_schema.png")

## 2. Sample Subgraph

**Objective:** show what the graph looks like around a handful of real patients.

**Implementation:** sample 6 `Patient` nodes, collect their `Image`, `Finding`, and `ViewPosition` neighbors, and lay out the resulting subgraph with `spring_layout`.

**Expected output:** `02_sample_subgraph.png`, a labeled, color-coded subgraph.

**Discussion:** `random.seed(42)` (set in the imports cell above) makes the sampled patients reproducible across runs.

In [ ]:
patient_nodes = [n for n, d in G.nodes(data=True) if d.get("node_type") == "Patient"]
sample_patients = random.sample(patient_nodes, 6)

sub_nodes = set(sample_patients)
for p in sample_patients:
    for img in G.successors(p):
        sub_nodes.add(img)
        for nbr in G.successors(img):
            sub_nodes.add(nbr)

SG = G.subgraph(sub_nodes).copy()

fig, ax = plt.subplots(figsize=(13, 10))
pos = nx.spring_layout(SG, seed=7, k=0.6, iterations=60)

for nt, color in COLORS.items():
    nodelist = [n for n, d in SG.nodes(data=True) if d.get("node_type") == nt]
    sizes = {"Patient": 500, "Image": 220, "Finding": 650, "ViewPosition": 450}[nt]
    nx.draw_networkx_nodes(SG, pos, nodelist=nodelist, node_color=color,
                            node_size=sizes, alpha=0.9, ax=ax, edgecolors="white", linewidths=1)

edge_colors = {"HAS_IMAGE": "#4C72B0", "HAS_FINDING": "#C44E52",
               "TAKEN_IN_VIEW": "#8172B2", "CO_OCCURS_WITH": "#999999"}
for rel, color in edge_colors.items():
    elist = [(u, v) for u, v, d in SG.edges(data=True) if d.get("relation") == rel]
    nx.draw_networkx_edges(SG, pos, edgelist=elist, edge_color=color, alpha=0.5,
                            arrows=True, arrowsize=8, width=1.2, ax=ax)

labels = {}
for n, d in SG.nodes(data=True):
    if d["node_type"] == "Patient":
        labels[n] = f"Patient {d['patient_id']}"
    elif d["node_type"] == "Finding":
        labels[n] = d["name"]
    elif d["node_type"] == "ViewPosition":
        labels[n] = d["name"]
nx.draw_networkx_labels(SG, pos, labels=labels, font_size=8, ax=ax)

legend_elems = [Line2D([0], [0], marker="o", color="w", label=nt,
                        markerfacecolor=c, markersize=12) for nt, c in COLORS.items()]
ax.legend(handles=legend_elems, loc="upper left", fontsize=11, title="Node type")
ax.set_title(f"Sample Subgraph — {len(sample_patients)} Patients and Their Connected Images/Findings/Views",
             fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "02_sample_subgraph.png"), dpi=180, bbox_inches="tight")
plt.close()
print("Saved 02_sample_subgraph.png")

## 3. Finding Co-occurrence Network

**Objective:** visualize which diseases tend to be recorded together on the same image.

**Implementation:** build an undirected `Finding`-only graph from `finding_pair_counts`, sizing nodes by how many images carry that finding and edges by co-occurrence count.

**Expected output:** `03_finding_cooccurrence.png`.

**Discussion:** this view is reused conceptually in the Precision Medicine KG section later, which imports these exact co-occurrence counts as `COMORBID_WITH` edges.

In [ ]:
with open(os.path.join(KG_DIR, "finding_pair_counts.pkl"), "rb") as fh:
    pair_counts = pickle.load(fh)

FG = nx.Graph()
for n, d in G.nodes(data=True):
    if d.get("node_type") == "Finding" and d["name"] != "No Finding":
        FG.add_node(d["name"])

for (f1, f2), w in pair_counts.items():
    FG.add_edge(f1, f2, weight=w)

# Node size ~ total number of images with that finding (count HAS_FINDING in-edges)
finding_image_count = {f: 0 for f in FG.nodes()}
for u, v, d in G.edges(data=True):
    if d.get("relation") == "HAS_FINDING":
        fname = G.nodes[v]["name"]
        if fname in finding_image_count:
            finding_image_count[fname] += 1

fig, ax = plt.subplots(figsize=(11, 9))
pos = nx.circular_layout(FG)

sizes = [300 + 25 * finding_image_count[n] ** 0.5 for n in FG.nodes()]
weights = np.array([FG[u][v]["weight"] for u, v in FG.edges()])
widths = 0.5 + 6 * (weights / weights.max())

nx.draw_networkx_edges(FG, pos, width=widths, edge_color="#C44E52", alpha=0.4, ax=ax)
nx.draw_networkx_nodes(FG, pos, node_size=sizes, node_color="#C44E52", alpha=0.85,
                        edgecolors="white", linewidths=1.5, ax=ax)
nx.draw_networkx_labels(FG, pos, font_size=10, font_weight="bold", ax=ax)

ax.set_title("Finding Co-occurrence Network\n(node size = # images with finding, edge width = co-occurrence count)",
              fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "03_finding_cooccurrence.png"), dpi=180, bbox_inches="tight")
plt.close()
print("Saved 03_finding_cooccurrence.png")

# Export ChestX-ray14 Knowledge Graph to GraphML

**Objective:** produce a portable, tool-agnostic artifact of `G` that opens in Gephi, Neo4j, yEd, or any other graph tool — not just this notebook.

**Implementation:** reload `G`, coerce Python `bool` attributes to `str` (GraphML has no native boolean type), then `nx.write_graphml`.

**Expected output:** `chestxray14_knowledge_graph.graphml` under `OUTPUT_DIR`.

**Discussion:** the explicit bool->str cast avoids the notebook's most common GraphML export pitfall — a silent `TypeError`/misexport for attributes GraphML can't natively type.

In [ ]:
# Reload G (lets this export section run standalone from a fresh kernel)
with open(os.path.join(KG_DIR, "graph.gpickle"), "rb") as fh:
    G = pickle.load(fh)

GraphML requires simple, GraphML-typed attribute values (str/int/float/bool).
Our node/edge attrs are already primitives, but MultiDiGraph edge keys with
string form (e.g. "HAS_FINDING::Mass") are fine as GraphML supports multigraphs.
`is_no_finding` is bool -> GraphML supports bool via yfiles-style attr; networkx
handles bool by casting to string automatically-safe, but let's be explicit.

In [ ]:
for n, data in G.nodes(data=True):
    for k, v in list(data.items()):
        if isinstance(v, bool):
            data[k] = str(v)

In [ ]:
out_path = os.path.join(OUTPUT_DIR, "chestxray14_knowledge_graph.graphml")
nx.write_graphml(G, out_path)

print(f"Exported GraphML to {out_path}")

In [ ]:
size_mb = os.path.getsize(out_path) / (1024 * 1024)
print(f"File size: {size_mb:.1f} MB")

# Second Multimedia Pipeline — Direct Image-Based Deep Learning Inference

Everything above this point (Sections 13–21, the knowledge graph build, its visualizations, and the
GraphML export) works from **tabular derivatives** of the dataset: a pre-extracted feature CSV for
the classical ML models, and the metadata CSV for the knowledge graph. Neither of those pipelines
ever opens a single JPEG/PNG file.

This second pipeline is different on purpose: it is a **multimedia pipeline that consumes the raw
ChestX-ray14 image files directly**. There is no intermediate `image_features_dataset.csv`, no
hand-engineered feature table — a pretrained deep convolutional network reads pixels straight off
disk and produces two kinds of *semantic outputs* in memory:

1. **Disease predictions** — per-pathology probabilities, produced by the network's classification
   head.
2. **Image embeddings** — dense vector representations from the network's penultimate layer,
   produced by the same forward pass.

Per the task scope, this pipeline **stops at those semantic outputs**. It does not write a features
CSV, and it does not construct or touch the knowledge graph built earlier in this notebook.

## Step 1 — Imports

All imports for the direct-image pipeline (`torch`, `torchvision`/`torchxrayvision`, `PIL`, `sklearn.decomposition.PCA`) — plus the `HAVE_XRV` capability flag and the `DEVICE` (GPU/CPU) selection — are already established in **Global Setup** at the top of the notebook, so this step has no code cell of its own.

**Objective:** confirm this pipeline's imports are already covered by Global Setup.

**Implementation:** `torch`, `torch.nn.functional`, `PIL.Image`, `torchxrayvision` (with a `torchvision` fallback), `HAVE_XRV`, and `DEVICE` were all established once at the top of the notebook.

**Discussion:** no code cell needed here — see Global Setup.

## Step 2 — Point directly at the raw image files

No CSV of pre-computed features is read here. `IMAGE_DIR` should point at the folder of raw
ChestX-ray14 images (`images_001/images`, `images_002/images`, ... in the original NIH release, or
a flattened folder of the same PNGs). The metadata CSV is opened only to grab filenames and, where
available, ground-truth labels for **display/context** in the visualizations later — it is never
used to build model input features.

**Objective:** point the pipeline at a folder of raw image files instead of a features CSV.

**Implementation:** `IMAGE_DIR`/`METADATA_CSV` come from Global Setup; only `N_SAMPLE_IMAGES` (how many images this demo run processes) is section-specific.

**Expected output:** a count of discovered images and the sampled subset.

**Discussion:** `gt_lookup` is populated only for titling plots later — it never reaches the model or the multimedia knowledge graph's construction logic.

In [ ]:
# S2 — Configuration: raw image directory + a light metadata lookup for display only
# IMAGE_DIR, METADATA_CSV, and RANDOM_STATE are already set in Global Setup (edit them there)
N_SAMPLE_IMAGES = 12

image_paths = sorted(
    glob.glob(os.path.join(IMAGE_DIR, "*.png")) + glob.glob(os.path.join(IMAGE_DIR, "*.jpg"))
)
assert len(image_paths) > 0, (
    f"No images found under {IMAGE_DIR}. Point IMAGE_DIR at the folder containing the raw "
    f"ChestX-ray14 image files (this pipeline reads pixels directly, not a features CSV)."
)

rng = np.random.RandomState(RANDOM_STATE)
sample_paths = list(rng.choice(image_paths, size=min(N_SAMPLE_IMAGES, len(image_paths)), replace=False))

# Optional: ground-truth labels purely for titling the visualizations later, not for model input
gt_lookup = {}
if os.path.exists(METADATA_CSV):
    meta = pd.read_csv(METADATA_CSV)
    id_col = meta.columns[0]      # "Image Index" in the original NIH file
    label_col = meta.columns[1]   # "Finding Labels"
    gt_lookup = dict(zip(meta[id_col], meta[label_col]))

print(f"✅ Found {len(image_paths)} raw image files under {IMAGE_DIR}")
print(f"✅ Sampled {len(sample_paths)} images for this run")

## Step 3 — Load a pretrained deep learning model for medical images

`torchxrayvision`'s `DenseNet` (`densenet121-res224-all`) was trained across several large public
chest X-ray datasets, NIH ChestX-ray14 among them, and exposes an 18-pathology sigmoid output head
plus a 1024-d feature extractor — a single pretrained model that already understands chest
radiographs, no training loop required here. If the package isn't installed, we fall back to an
ImageNet-pretrained DenseNet121 from `torchvision` purely as a demonstration of the pipeline
mechanics (its predictions would need fine-tuning on CXR data to be clinically meaningful).

**Objective:** load a CXR-specific pretrained network instead of training one from scratch.

**Implementation:** `torchxrayvision`'s DenseNet if available, else an ImageNet-pretrained `torchvision` DenseNet121 as a mechanics-only fallback.

**Expected output:** `dl_model`, `PATHOLOGIES` (its output vocabulary), and `IMG_SIZE`.

**Discussion:** the fallback path's predictions are not clinically meaningful without fine-tuning — it exists so the rest of the pipeline still runs when `torchxrayvision` isn't installed.

In [ ]:
# S3 — Load the pretrained model
if HAVE_XRV:
    dl_model = xrv.models.DenseNet(weights="densenet121-res224-all")
    PATHOLOGIES = dl_model.pathologies                 # model-provided disease vocabulary
    IMG_SIZE = 224
else:
    dl_model = torchvision.models.densenet121(weights=torchvision.models.DenseNet121_Weights.IMAGENET1K_V1)
    dl_model.classifier = torch.nn.Linear(dl_model.classifier.in_features, 14)
    torch.nn.init.xavier_uniform_(dl_model.classifier.weight)
    PATHOLOGIES = DISEASE_NAMES if "DISEASE_NAMES" in dir() else [f"Disease_{i}" for i in range(14)]
    IMG_SIZE = 224

dl_model.eval().to(DEVICE)

n_params = sum(p.numel() for p in dl_model.parameters())
print(f"✅ Loaded pretrained model ({'torchxrayvision DenseNet121' if HAVE_XRV else 'torchvision DenseNet121 (fallback)'})")
print(f"   Parameters: {n_params:,}")
print(f"   Pathology vocabulary ({len(PATHOLOGIES)}): {PATHOLOGIES}")

## Step 4 — Direct image preprocessing (raw pixels in, tensor out — no CSV in between)

Each image is read straight from disk with PIL, resized, converted to a tensor, and normalized —
entirely in memory. Contrast this with the earlier pipeline's CSV, which required a *separate*
offline pass that ran a CNN over every image once and wrote a 2048-d vector per row to disk before
any modeling could start. Here, preprocessing and inference happen together, per image, on demand.

**Objective:** turn a raw image file into a model-ready tensor with no CSV round-trip.

**Implementation:** `PIL.Image.open` -> resize -> normalize -> `torch.Tensor`, entirely in memory.

**Expected output:** `load_and_preprocess_image()`, reused by every inference cell below.

**Discussion:** contrast with Sections 13-21's `image_features_dataset.csv`, which required a separate offline pass over every image before any modeling could start.

In [ ]:
# S4 — Load + preprocess one raw image into a model-ready tensor (no disk features involved)
def load_and_preprocess_image(path, img_size=IMG_SIZE):
    img = Image.open(path).convert("L")                          # chest X-rays are grayscale
    img = img.resize((img_size, img_size), Image.BILINEAR)
    arr = np.asarray(img, dtype=np.float32)

    if HAVE_XRV:
        arr = xrv.datasets.normalize(arr, 255)                    # maps to xrv's [-1024, 1024] convention
        tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    else:
        arr = (arr / 255.0 - 0.5) / 0.5
        tensor = torch.from_numpy(arr).unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)  # [1, 3, H, W]

    return tensor.to(DEVICE)

print("✅ load_and_preprocess_image() ready — pixels go from disk to tensor with no CSV round-trip")

## Step 5 — Generate disease predictions directly from pixels

For every sampled image, the tensor is pushed through the pretrained network's forward pass and the
sigmoid output head is read off as per-pathology probabilities. The result lives in a Python dict /
DataFrame in memory for this notebook session only — nothing is written to a features CSV.

**Objective:** get per-pathology probabilities straight from pixels.

**Implementation:** one forward pass per sampled image; sigmoid over the model's output head.

**Expected output:** `image_predictions` (dict) and `pred_df` (its tabular view).

**Discussion:** nothing here is written to disk — these predictions exist only in memory for this session, by design (see "Stopping point" below).

In [ ]:
# S5 — Run inference: raw image -> pretrained model -> disease probabilities
image_predictions = {}   # image_path -> {pathology: probability}

with torch.no_grad():
    for path in sample_paths:
        img_tensor = load_and_preprocess_image(path)
        logits = dl_model(img_tensor)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        image_predictions[path] = dict(zip(PATHOLOGIES, probs))

pred_df = pd.DataFrame(image_predictions).T
pred_df.index = [os.path.basename(p) for p in pred_df.index]

print(f"✅ Generated predictions for {len(pred_df)} images across {len(PATHOLOGIES)} pathologies")
pred_df.round(3).head()

## Step 6 — Extract image embeddings directly from raw pixels

The same forward pass that produced predictions also exposes the network's penultimate
representation — a dense embedding capturing learned visual structure, independent of any specific
disease label. `torchxrayvision` exposes this via `.features()`; for the generic fallback model we
hook the pooled output before the classifier.

**Objective:** capture the network's learned visual representation, not just its disease predictions.

**Implementation:** the same forward pass exposes the penultimate-layer activations via `.features()`.

**Expected output:** `image_embeddings`, one dense vector per sampled image.

**Discussion:** these embeddings are what power `VISUALLY_SIMILAR_TO` and `VisualCluster` in the multimedia knowledge graph built later — relationships a metadata CSV can't express.

In [ ]:
# S6 — Extract a dense embedding vector per raw image (in-memory, not written to CSV)
image_embeddings = {}   # image_path -> 1D numpy embedding vector

with torch.no_grad():
    for path in sample_paths:
        img_tensor = load_and_preprocess_image(path)
        if HAVE_XRV:
            feats = dl_model.features(img_tensor)                 # [1, 1024]
        else:
            feats = dl_model.features(img_tensor)
            feats = F.adaptive_avg_pool2d(feats, (1, 1)).flatten(1)  # [1, 1024]
        image_embeddings[path] = feats.cpu().numpy().flatten()

embedding_dim = next(iter(image_embeddings.values())).shape[0]
print(f"✅ Extracted embeddings for {len(image_embeddings)} images, each {embedding_dim}-dimensional")
print("   (kept in memory as a dict for this session — no embeddings CSV is written)")

## Step 6b — Attention / saliency maps (Grad-CAM)

**Objective:** produce a fifth kind of semantic output directly from the pretrained model — a
spatial map of *where in the image* it is looking when it predicts a given pathology — completing
the set (predictions, confidence scores, embeddings, semantic features, attention maps).

**Implementation:** Grad-CAM is computed from the same backbone already loaded in Step 3. It hooks
the convolutional feature map returned by `dl_model.features(...)`, backpropagates the score for
each image's top predicted pathology, and weights the feature channels by their gradients to build
a coarse localization heatmap — no extra model download or dependency required. This works for the
DenseNet-style backbone used here (`torchxrayvision` or the `torchvision` fallback from Step 3); a
Vision-Transformer or CLIP-style backbone would instead use attention-weight rollout, since it has
no convolutional feature map to hook.

**Expected output:** `image_attention_maps` — an in-memory dict of `image_path -> 2D heatmap array`
— plus a grid of the sampled X-rays with their Grad-CAM heatmap overlaid, highlighting the region
that most influenced each image's top prediction.

**Discussion:** confidence scores (Step 5's per-pathology probabilities) say *how sure* the model
is; attention maps say *where* that judgment came from. Together they make the model's semantic
output substantially more auditable than a bare probability vector — an important property to carry
into the knowledge graph's own explainability discussion later in this notebook.

In [ ]:
# S6b — Grad-CAM: gradient-weighted class activation maps from the same backbone as Steps 3/5/6
def compute_gradcam(model, img_tensor, target_idx):
    """Assumes a DenseNet-style features -> ReLU -> global-avg-pool -> classifier backbone,
    matching the model loaded in Step 3. Swap in attention-rollout here for a ViT/CLIP backbone."""
    feats = model.features(img_tensor)          # [1, C, h, w]
    feats.retain_grad()
    pooled = F.adaptive_avg_pool2d(F.relu(feats), (1, 1)).flatten(1)
    logits = model.classifier(pooled)
    score = logits[0, target_idx]

    model.zero_grad()
    score.backward(retain_graph=False)

    grads = feats.grad                            # [1, C, h, w]
    weights = grads.mean(dim=(2, 3), keepdim=True)  # global-avg-pool the gradients per channel
    cam = F.relu((weights * feats).sum(dim=1, keepdim=True)).squeeze().detach().cpu().numpy()

    cam = cam - cam.min()
    if cam.max() > 0:
        cam = cam / cam.max()
    return cam


image_attention_maps = {}   # image_path -> 2D numpy heatmap, same target as each image's top prediction

for path in sample_paths:
    img_tensor = load_and_preprocess_image(path)
    top_finding = max(image_predictions[path].items(), key=lambda kv: kv[1])[0]
    target_idx = PATHOLOGIES.index(top_finding)
    image_attention_maps[path] = compute_gradcam(dl_model, img_tensor, target_idx)

print(f"✅ Computed Grad-CAM attention maps for {len(image_attention_maps)} images")
print("   (kept in memory as a dict for this session — no attention-map CSV is written)")

In [ ]:
# S6c — Visualize a few Grad-CAM overlays
n_show = min(8, len(sample_paths))
n_cols = 4
n_rows = int(np.ceil(n_show / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.4 * n_cols, 3.8 * n_rows))
axes = np.array(axes).flatten()

for i, path in enumerate(sample_paths[:n_show]):
    ax = axes[i]
    img = Image.open(path).convert("L").resize((IMG_SIZE, IMG_SIZE))
    cam = image_attention_maps[path]
    cam_resized = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE)))

    ax.imshow(img, cmap="gray")
    ax.imshow(cam_resized, cmap="jet", alpha=0.4)

    top_finding, top_prob = max(image_predictions[path].items(), key=lambda kv: kv[1])
    ax.set_title(f"{os.path.basename(path)}\n{top_finding}: {top_prob:.2f}", fontsize=8)
    ax.axis("off")

for j in range(n_show, len(axes)):
    axes[j].axis("off")

plt.suptitle("Grad-CAM attention maps — top predicted pathology per image", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Step 7 — Visualize model predictions

**Objective:** make the model's predictions inspectable rather than a bare probability table.

**Implementation:** an image grid with top-3 predictions as titles, a bar chart of average per-pathology probability, and a 2D PCA projection of the embeddings.

**Expected output:** three plots.

**Discussion:** the PCA projection is a quick way to eyeball whether the embedding space clusters images by predicted pathology before it's used more formally by `VisualCluster` nodes below.

In [ ]:
# S7a — Grid of sampled X-rays with their top predicted pathologies as titles
n_cols = 4
n_rows = int(np.ceil(len(sample_paths) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.4 * n_cols, 3.8 * n_rows))
axes = np.array(axes).flatten()

for i, path in enumerate(sample_paths):
    ax = axes[i]
    img = Image.open(path).convert("L")
    ax.imshow(img, cmap="gray")

    probs = image_predictions[path]
    top3 = sorted(probs.items(), key=lambda kv: kv[1], reverse=True)[:3]
    title = "\n".join(f"{name}: {p:.2f}" for name, p in top3)
    fname = os.path.basename(path)
    gt = gt_lookup.get(fname, "")
    ax.set_title(f"{fname}\n{title}" + (f"\nGT: {gt}" if gt else ""), fontsize=8)
    ax.axis("off")

for j in range(len(sample_paths), len(axes)):
    axes[j].axis("off")

plt.suptitle("Direct-from-pixels predictions — pretrained CXR model", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# S7b — Per-pathology predicted probability, averaged across the sampled images
mean_probs = pred_df.mean(axis=0).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
mean_probs.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_ylabel("Mean predicted probability")
ax.set_title("Average predicted disease probability across sampled images")
ax.set_ylim(0, 1)
plt.xticks(rotation=60, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# S7c — 2D projection of the raw-pixel embeddings, colored by each image's top predicted pathology
emb_matrix = np.stack([image_embeddings[p] for p in sample_paths])
top_labels = [max(image_predictions[p].items(), key=lambda kv: kv[1])[0] for p in sample_paths]

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
emb_2d = pca_2d.fit_transform(emb_matrix)

fig, ax = plt.subplots(figsize=(7, 6))
unique_labels = sorted(set(top_labels))
cmap = plt.get_cmap("tab10")
for i, lbl in enumerate(unique_labels):
    idx = [j for j, l in enumerate(top_labels) if l == lbl]
    ax.scatter(emb_2d[idx, 0], emb_2d[idx, 1], label=lbl, s=90, color=cmap(i % 10), edgecolors="white")

ax.set_xlabel("PCA-1")
ax.set_ylabel("PCA-2")
ax.set_title("Image embeddings (from raw pixels), colored by top predicted pathology")
ax.legend(fontsize=8, loc="best")
plt.tight_layout()
plt.show()

## How multimedia objects are processed directly (no CSV features)

The two pipelines in this notebook take genuinely different data paths:

| | First pipeline (Sections 13–21) | Second pipeline (this section) |
|---|---|---|
| Input | `image_features_dataset.csv` — one row per image with pre-extracted `feature_*` columns | Raw `.png` / `.jpg` files under `IMAGE_DIR`, read directly |
| Where the CNN runs | Once, offline, in a separate data-pipeline notebook, writing vectors to disk | Inline, in this notebook, at prediction time |
| Intermediate artifact | A persisted CSV of numeric features | None — a tensor exists only in memory for the duration of one forward pass |
| Models used | Classical ML (Logistic Regression, Random Forest, SVM) on the CSV's frozen features | The pretrained deep network itself, end-to-end on pixels |
| Output | Predictions from classical models operating on tabular features | Predictions *and* embeddings produced directly by the deep network's own forward pass |

Concretely, each raw image follows: **file on disk → `PIL.Image.open` → resize/normalize →
`torch.Tensor` → one forward pass through the pretrained CNN → sigmoid layer (predictions) and
penultimate-layer activations (embeddings)**. All of that happens inside `load_and_preprocess_image`
and the inference loops in Steps 4–6, per image, with no CSV read or write anywhere in between. The
model consumes the *multimedia object itself* — the image — rather than a hand-off table of numbers
someone else computed earlier.

## Stopping point

This pipeline stops here, at its semantic outputs:

- `image_predictions` — per-image, per-pathology probabilities, produced directly from pixels
- `image_embeddings` — per-image dense feature vectors, produced directly from pixels

No features CSV was generated. The structured knowledge graph earlier in this notebook was
constructed exclusively from `Data_Entry_2017_v2020.csv` metadata and remains untouched by this
image-based pipeline. The next section builds a **separate** knowledge graph, sourced directly from
`image_predictions` and `image_embeddings` above.

# Knowledge Graph from Direct Multimedia Semantic Outputs

The structured knowledge graph built earlier in this notebook (node types `Patient`, `Image`,
`Finding`, `ViewPosition`) is derived entirely from **columns in a spreadsheet** —
`Data_Entry_2017_v2020.csv`. Every node and edge there is a fact someone already recorded in a
table: which patient owns which image, which view position a scan was taken in, which finding
labels a radiologist logged.

This section builds a **second, independent knowledge graph** — sourced exclusively from the
*semantic outputs of the deep learning model* computed above: `image_predictions` and
`image_embeddings`. Nothing here comes from `Data_Entry_2017_v2020.csv`; ground-truth labels
(`gt_lookup`) are deliberately excluded from the graph's structure so the ontology reflects only
what the model itself produced from pixels.

## Step 8 — Ontology design for the multimedia knowledge graph

A metadata table only ever gives you *categorical, hand-recorded* relationships — an image belongs
to a patient, was taken in a view, carries a finding label a person typed in. Direct pixel processing
gives you something a spreadsheet cannot: **probabilistic** relationships (how *confident* the model
is in a finding) and **learned-similarity** relationships (which images look alike in the network's
own representation space, discovered by comparing embeddings — not by matching any shared column
value). The ontology below is designed around those two properties.

**Node types (3):**
1. `Image` — id: image filename; attrs: none inferred from metadata, only what pixels+model produced
2. `PredictedFinding` — id: pathology name from the model's own output vocabulary (`PATHOLOGIES`)
3. `VisualCluster` — id: cluster index from k-means over the embedding space; a node type with **no
   equivalent in the structured graph**, since it doesn't correspond to any spreadsheet column — it
   only exists because embeddings carve the images into visually coherent groups

**Relationship types (4):**
1. `(Image) -[:PREDICTED_FINDING]-> (PredictedFinding)` — attr: `probability` (soft-weighted, not a
   binary flag; only emitted above `PRED_THRESHOLD`)
2. `(Image) -[:VISUALLY_SIMILAR_TO]-> (Image)` — attr: `similarity` (cosine similarity between
   embeddings); connects images to their nearest neighbors in representation space, with **no
   basis in any metadata field**
3. `(Image) -[:BELONGS_TO_CLUSTER]-> (VisualCluster)` — derived from k-means over embeddings
4. `(PredictedFinding) -[:CO_PREDICTED_WITH]-> (PredictedFinding)` — derived, symmetric, attr:
   `weight` (how often two findings are predicted together above threshold on the same image)

**Imports:** `itertools`, `pickle`, `Counter`, `networkx`, plot helpers, `sklearn.cluster.KMeans`, and `sklearn.metrics.pairwise.cosine_similarity` are already available from Global Setup.

**Objective:** design a schema around what a pretrained model's outputs can express, distinct from what a metadata CSV can express.

**Implementation:** see the ontology below — 3 node types, 4 relationship types.

**Expected output:** none (design-only markdown); realized by Step 9's code.

**Discussion:** the key design choice is *soft* edges (`probability`, `similarity`) instead of the structured graph's categorical, present/absent edges.

## Step 9 — Build the graph with NetworkX

Built entirely from `image_predictions`, `image_embeddings`, and `PATHOLOGIES` — the in-memory
objects the deep learning pipeline produced above. No file is read in this cell.

**Objective:** materialize the Step 8 ontology as a `networkx.MultiDiGraph`.

**Implementation:** built in four passes — node creation, `PREDICTED_FINDING` edges, `VISUALLY_SIMILAR_TO` edges, and `VisualCluster` assignment — each in its own cell below.

**Expected output:** `mmG`, in memory.

**Discussion:** unlike the structured graph, every attribute here traces to a specific model output, never a spreadsheet column.

In [ ]:
# S9a — Configuration for graph construction
PRED_THRESHOLD = 0.5        # minimum predicted probability to add a PREDICTED_FINDING edge
K_NEAREST = 3                # neighbors per image for VISUALLY_SIMILAR_TO
N_CLUSTERS = min(4, len(sample_paths))

mmG = nx.MultiDiGraph(name="ChestXray14_MultimediaKnowledgeGraph")

img_ids = [os.path.basename(p) for p in sample_paths]
path_by_id = dict(zip(img_ids, sample_paths))

# --- Image nodes — attrs come only from the model's own outputs, not from any CSV ---
for img_id, path in zip(img_ids, sample_paths):
    top_finding, top_prob = max(image_predictions[path].items(), key=lambda kv: kv[1])
    mmG.add_node(
        f"Image::{img_id}",
        node_type="Image",
        image_id=img_id,
        top_predicted_finding=top_finding,
        top_predicted_probability=float(top_prob),
        embedding_dim=int(image_embeddings[path].shape[0]),
    )

# --- PredictedFinding nodes — one per pathology in the model's own output vocabulary ---
for finding in PATHOLOGIES:
    mmG.add_node(
        f"PredictedFinding::{finding}",
        node_type="PredictedFinding",
        name=finding,
    )

print(f"✅ Added {len(img_ids)} Image nodes and {len(PATHOLOGIES)} PredictedFinding nodes")

In [ ]:
# S9b — PREDICTED_FINDING edges (soft, probability-weighted — not a categorical flag)
finding_pair_counts_mm = {}   # (finding1, finding2) -> co-prediction weight

for img_id, path in zip(img_ids, sample_paths):
    probs = image_predictions[path]
    predicted_above_threshold = [f for f, p in probs.items() if p >= PRED_THRESHOLD]

    for finding in predicted_above_threshold:
        mmG.add_edge(
            f"Image::{img_id}", f"PredictedFinding::{finding}",
            key=f"PREDICTED_FINDING::{finding}",
            relation="PREDICTED_FINDING",
            probability=float(probs[finding]),
        )

    for f1, f2 in itertools.combinations(sorted(predicted_above_threshold), 2):
        finding_pair_counts_mm[(f1, f2)] = finding_pair_counts_mm.get((f1, f2), 0) + 1

for (f1, f2), w in finding_pair_counts_mm.items():
    mmG.add_edge(f"PredictedFinding::{f1}", f"PredictedFinding::{f2}",
                 key="CO_PREDICTED_WITH", relation="CO_PREDICTED_WITH", weight=w)
    mmG.add_edge(f"PredictedFinding::{f2}", f"PredictedFinding::{f1}",
                 key="CO_PREDICTED_WITH", relation="CO_PREDICTED_WITH", weight=w)

n_pf_edges = sum(1 for _, _, d in mmG.edges(data=True) if d["relation"] == "PREDICTED_FINDING")
print(f"✅ Added {n_pf_edges} PREDICTED_FINDING edges (threshold={PRED_THRESHOLD})")
print(f"✅ Added {len(finding_pair_counts_mm) * 2} CO_PREDICTED_WITH edges")

In [ ]:
# S9c — VISUALLY_SIMILAR_TO edges, derived purely from embedding-space cosine similarity
emb_matrix_all = np.stack([image_embeddings[p] for p in sample_paths])
sim_matrix = cosine_similarity(emb_matrix_all)

n_sim_edges = 0
for i, img_id_i in enumerate(img_ids):
    sims = [(j, sim_matrix[i, j]) for j in range(len(img_ids)) if j != i]
    sims.sort(key=lambda t: t[1], reverse=True)
    for j, sim in sims[:K_NEAREST]:
        mmG.add_edge(
            f"Image::{img_id_i}", f"Image::{img_ids[j]}",
            key="VISUALLY_SIMILAR_TO", relation="VISUALLY_SIMILAR_TO",
            similarity=float(sim),
        )
        n_sim_edges += 1

print(f"✅ Added {n_sim_edges} VISUALLY_SIMILAR_TO edges (k={K_NEAREST} nearest neighbors per image)")

In [ ]:
# S9d — VisualCluster nodes + BELONGS_TO_CLUSTER edges, derived by k-means over embeddings
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
cluster_assignments = kmeans.fit_predict(emb_matrix_all)

for c in range(N_CLUSTERS):
    members = [img_ids[i] for i in range(len(img_ids)) if cluster_assignments[i] == c]
    mmG.add_node(
        f"VisualCluster::{c}",
        node_type="VisualCluster",
        cluster_id=int(c),
        n_members=len(members),
    )

for img_id, c in zip(img_ids, cluster_assignments):
    mmG.add_edge(
        f"Image::{img_id}", f"VisualCluster::{c}",
        key="BELONGS_TO_CLUSTER", relation="BELONGS_TO_CLUSTER",
    )

print(f"✅ Added {N_CLUSTERS} VisualCluster nodes and {len(img_ids)} BELONGS_TO_CLUSTER edges")
print(f"✅ Graph built: {mmG.number_of_nodes()} nodes, {mmG.number_of_edges()} edges")

## Step 10 — Visualize the multimedia knowledge graph

**Objective:** confirm `mmG`'s structure visually, the same way Section "Visualize the ChestX-ray14 Knowledge Graph" did for `G`.

**Implementation:** schema diagram, full node/edge-colored graph, and an image-similarity-only subgraph.

**Expected output:** three plots, shown inline (not saved to disk, unlike the structured graph's visualizations).

**Discussion:** the similarity-only view isolates the one relationship type (`VISUALLY_SIMILAR_TO`) with literally no equivalent in the structured graph.

In [ ]:
# S10a — Ontology / schema diagram
COLORS_MM = {
    "Image": "#55A868",
    "PredictedFinding": "#C44E52",
    "VisualCluster": "#DD8452",
}

fig, ax = plt.subplots(figsize=(10, 6.5))

S = nx.MultiDiGraph()
for nt in COLORS_MM:
    S.add_node(nt)

pos_schema = {"Image": (0, 0), "PredictedFinding": (5, 1.2), "VisualCluster": (5, -1.2)}

for nt, (x, y) in pos_schema.items():
    ax.add_patch(mpatches.FancyBboxPatch(
        (x - 1.1, y - 0.4), 2.2, 0.8,
        boxstyle="round,pad=0.05,rounding_size=0.1",
        linewidth=2, edgecolor=COLORS_MM[nt], facecolor=COLORS_MM[nt] + "33"))
    ax.text(x, y, nt, ha="center", va="center", fontsize=12, fontweight="bold", color=COLORS_MM[nt])

def draw_edge(a, b, label, curve=0.0, self_loop=False):
    xa, ya = pos_schema[a]; xb, yb = pos_schema[b]
    if self_loop:
        ax.annotate("", xy=(xb + 1.1, yb + 0.5), xytext=(xb + 1.1, yb - 0.5),
                    arrowprops=dict(arrowstyle="-|>", color="#555555", lw=1.8,
                                     connectionstyle="arc3,rad=1.4"))
        ax.text(xb + 2.4, yb, label, fontsize=9.5, ha="left", va="center", style="italic")
    else:
        ax.annotate("", xy=(xb - 1.1, yb), xytext=(xa + 1.1, ya),
                    arrowprops=dict(arrowstyle="-|>", color="#555555", lw=1.8,
                                     connectionstyle=f"arc3,rad={curve}"))
        mx, my = (xa + xb) / 2, (ya + yb) / 2 + curve * 1.5
        ax.text(mx, my + 0.25, label, fontsize=9.5, ha="center", va="center", style="italic")

draw_edge("Image", "PredictedFinding", "PREDICTED_FINDING (prob)", curve=0.12)
draw_edge("Image", "VisualCluster", "BELONGS_TO_CLUSTER", curve=-0.12)
draw_edge("Image", "Image", "VISUALLY_SIMILAR_TO (sim)", self_loop=True)
draw_edge("PredictedFinding", "PredictedFinding", "CO_PREDICTED_WITH (weight)", self_loop=True)

ax.set_xlim(-2.5, 9); ax.set_ylim(-2.5, 2.5)
ax.axis("off")
ax.set_title("Multimedia Knowledge Graph — Ontology (Schema)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "mm_01_ontology_schema.png"), dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# S10b — Full graph, node-type colored, edge-type colored
fig, ax = plt.subplots(figsize=(13, 10))
pos_mm = nx.spring_layout(mmG, seed=RANDOM_STATE, k=0.55, iterations=80)

size_map = {"Image": 420, "PredictedFinding": 650, "VisualCluster": 900}
for nt, color in COLORS_MM.items():
    nodelist = [n for n, d in mmG.nodes(data=True) if d.get("node_type") == nt]
    nx.draw_networkx_nodes(mmG, pos_mm, nodelist=nodelist, node_color=color,
                            node_size=size_map[nt], alpha=0.9, ax=ax,
                            edgecolors="white", linewidths=1)

edge_colors_mm = {
    "PREDICTED_FINDING": "#C44E52",
    "VISUALLY_SIMILAR_TO": "#4C72B0",
    "BELONGS_TO_CLUSTER": "#DD8452",
    "CO_PREDICTED_WITH": "#999999",
}
for rel, color in edge_colors_mm.items():
    elist = [(u, v) for u, v, d in mmG.edges(data=True) if d.get("relation") == rel]
    nx.draw_networkx_edges(mmG, pos_mm, edgelist=elist, edge_color=color, alpha=0.5,
                            arrows=True, arrowsize=8, width=1.2, ax=ax)

labels_mm = {}
for n, d in mmG.nodes(data=True):
    if d["node_type"] == "Image":
        labels_mm[n] = d["image_id"]
    elif d["node_type"] == "PredictedFinding":
        labels_mm[n] = d["name"]
    elif d["node_type"] == "VisualCluster":
        labels_mm[n] = f"Cluster {d['cluster_id']}"
nx.draw_networkx_labels(mmG, pos_mm, labels=labels_mm, font_size=7, ax=ax)

legend_elems = [Line2D([0], [0], marker="o", color="w", label=nt,
                        markerfacecolor=c, markersize=12) for nt, c in COLORS_MM.items()]
ax.legend(handles=legend_elems, loc="upper left", fontsize=10, title="Node type")
ax.set_title("Multimedia Knowledge Graph — built from model predictions + embeddings",
             fontsize=13, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "mm_02_full_graph.png"), dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# S10c — Visual-similarity network only (Image-Image edges), the relationship type with no
# equivalent in the structured, metadata-only knowledge graph
VSG = nx.Graph()
for img_id in img_ids:
    VSG.add_node(img_id)
for u, v, d in mmG.edges(data=True):
    if d.get("relation") == "VISUALLY_SIMILAR_TO":
        VSG.add_edge(mmG.nodes[u]["image_id"], mmG.nodes[v]["image_id"], weight=d["similarity"])

fig, ax = plt.subplots(figsize=(9, 8))
pos_vs = nx.spring_layout(VSG, seed=RANDOM_STATE, k=0.8)
weights = np.array([VSG[u][v]["weight"] for u, v in VSG.edges()])
widths = 0.5 + 5 * (weights - weights.min()) / (weights.max() - weights.min() + 1e-9)

nx.draw_networkx_edges(VSG, pos_vs, width=widths, edge_color="#4C72B0", alpha=0.5, ax=ax)
nx.draw_networkx_nodes(VSG, pos_vs, node_size=380, node_color="#55A868",
                        edgecolors="white", linewidths=1.2, ax=ax)
nx.draw_networkx_labels(VSG, pos_vs, font_size=7, ax=ax)
ax.set_title("Visual Similarity Network\n(edge width = cosine similarity between raw-pixel embeddings)",
             fontsize=12, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "mm_03_visual_similarity_network.png"), dpi=180, bbox_inches="tight")
plt.show()

## Step 11 — Graph statistics

**Objective:** get the same kind of numeric sanity-check Section 3 gave the structured graph.

**Implementation:** node/edge counts by type, density, degree stats, connected components, mean edge weights, and the most-connected images.

**Expected output:** printed statistics plus a degree-distribution histogram.

**Discussion:** these numbers feed directly into the cross-graph comparison sections later.

In [ ]:
# S11 — Node/edge counts, density, degree stats, connected components
node_type_counts_mm = Counter(nx.get_node_attributes(mmG, "node_type").values())
edge_type_counts_mm = Counter(d["relation"] for _, _, d in mmG.edges(data=True))

print("Node types:")
for k, v in node_type_counts_mm.items():
    print(f"  {k:18s} {v:,}")

print("\nRelationship types:")
for k, v in edge_type_counts_mm.items():
    print(f"  {k:20s} {v:,}")

n_nodes = mmG.number_of_nodes()
n_edges = mmG.number_of_edges()
density = nx.density(mmG)

undirected_mm = mmG.to_undirected()
n_components = nx.number_connected_components(undirected_mm)
largest_cc = max(nx.connected_components(undirected_mm), key=len)

degrees = [d for _, d in mmG.degree()]
avg_degree = float(np.mean(degrees))

print(f"\nTotal nodes: {n_nodes:,}   Total edges: {n_edges:,}")
print(f"Graph density: {density:.4f}")
print(f"Average node degree: {avg_degree:.2f}")
print(f"Connected components: {n_components}  (largest has {len(largest_cc)} nodes)")

pf_probs = [d["probability"] for _, _, d in mmG.edges(data=True) if d.get("relation") == "PREDICTED_FINDING"]
sim_scores = [d["similarity"] for _, _, d in mmG.edges(data=True) if d.get("relation") == "VISUALLY_SIMILAR_TO"]
print(f"\nMean PREDICTED_FINDING edge probability: {np.mean(pf_probs):.3f}")
print(f"Mean VISUALLY_SIMILAR_TO edge similarity: {np.mean(sim_scores):.3f}")

image_degrees = sorted(
    [(mmG.nodes[n]["image_id"], mmG.degree(n)) for n in mmG.nodes if mmG.nodes[n]["node_type"] == "Image"],
    key=lambda t: t[1], reverse=True,
)
print("\nMost-connected images (by total degree):")
for img_id, deg in image_degrees[:5]:
    print(f"  {img_id}: degree {deg}")

In [ ]:
# S11b — Degree distribution
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(degrees, bins=range(min(degrees), max(degrees) + 2), color="#4C72B0",
        edgecolor="white", align="left")
ax.set_xlabel("Node degree")
ax.set_ylabel("Number of nodes")
ax.set_title("Degree distribution — Multimedia Knowledge Graph")
plt.tight_layout()
plt.show()

## Step 12 — Export the graph

**Objective:** produce a portable artifact of `mmG`, mirroring the structured graph's GraphML export.

**Implementation:** cast `bool`/`np.bool_` attributes to `str` (GraphML has no native boolean type), then `nx.write_graphml`; also pickle `mmG` for reuse.

**Expected output:** `chestxray14_multimedia_knowledge_graph.graphml` under `OUTPUT_DIR`.

**Discussion:** the same GraphML pitfall (unhandled booleans) that Section "Export ChestX-ray14 Knowledge Graph to GraphML" guards against for `G` applies here too.

In [ ]:
# S12 — Export to GraphML (GraphML needs primitive-typed attrs; cast bools to str first)
mmG_export = mmG.copy()
for n, data in mmG_export.nodes(data=True):
    for k, v in list(data.items()):
        if isinstance(v, (bool, np.bool_)):
            data[k] = str(v)
for u, v, data in mmG_export.edges(data=True):
    for k, val in list(data.items()):
        if isinstance(val, (bool, np.bool_)):
            data[k] = str(val)

os.makedirs(OUTPUT_DIR, exist_ok=True)
mm_out_path = os.path.join(OUTPUT_DIR, "chestxray14_multimedia_knowledge_graph.graphml")
nx.write_graphml(mmG_export, mm_out_path)

with open(os.path.join(KG_DIR, "mm_graph.gpickle"), "wb") as fh:
    pickle.dump(mmG, fh)

size_kb = os.path.getsize(mm_out_path) / 1024
print(f"✅ Exported GraphML to {mm_out_path}")
print(f"   File size: {size_kb:.1f} KB")
print(f"   ({mmG.number_of_nodes()} nodes, {mmG.number_of_edges()} edges)")

## Step 13 — Validating the second Knowledge Graph

**Objective:** confirm the exported multimedia knowledge graph is structurally sound and faithful
to the ontology defined in Step 8, before treating it as a finished deliverable.

**Implementation:** the exported GraphML file is read back with `nx.read_graphml` and checked
against the in-memory `mmG` object for matching node/edge counts; the ontology's node types and
relationship types are checked for presence; and every `PREDICTED_FINDING` / `VISUALLY_SIMILAR_TO`
edge is checked for the numeric attribute (`probability` / `similarity`) its relationship type
requires.

**Expected output:** every assertion passes and a `✅ Validation passed` summary is printed. If the
GraphML round-trip lost an attribute (a common GraphML pitfall) or an edge is missing its expected
attribute, the corresponding assertion fails loudly instead of silently producing a broken graph.

**Discussion:** exporting to GraphML is not, by itself, proof the graph is correct — GraphML's
schema requires simple typed attributes, and a careless export can silently drop or stringify data.
This validation step is what actually earns the "second Knowledge Graph, completed and validated"
milestone requested for this section.

In [ ]:
# S13 — Validate the exported multimedia knowledge graph
mmG_reloaded = nx.read_graphml(mm_out_path)

assert mmG_reloaded.number_of_nodes() == mmG.number_of_nodes(), (
    f"Node count mismatch after GraphML round-trip: "
    f"{mmG_reloaded.number_of_nodes()} vs {mmG.number_of_nodes()}"
)
assert mmG_reloaded.number_of_edges() == mmG.number_of_edges(), (
    f"Edge count mismatch after GraphML round-trip: "
    f"{mmG_reloaded.number_of_edges()} vs {mmG.number_of_edges()}"
)

expected_node_types = {"Image", "PredictedFinding", "VisualCluster"}
found_node_types = set(nx.get_node_attributes(mmG, "node_type").values())
assert found_node_types == expected_node_types, (
    f"Node types don't match the Step 8 ontology: {found_node_types} vs {expected_node_types}"
)

expected_relations = {"PREDICTED_FINDING", "VISUALLY_SIMILAR_TO", "BELONGS_TO_CLUSTER", "CO_PREDICTED_WITH"}
found_relations = {d["relation"] for _, _, d in mmG.edges(data=True)}
assert found_relations == expected_relations, (
    f"Relationship types don't match the Step 8 ontology: {found_relations} vs {expected_relations}"
)

for u, v, d in mmG.edges(data=True):
    if d["relation"] == "PREDICTED_FINDING":
        assert "probability" in d, f"PREDICTED_FINDING edge {u}->{v} is missing its probability attribute"
    if d["relation"] == "VISUALLY_SIMILAR_TO":
        assert "similarity" in d, f"VISUALLY_SIMILAR_TO edge {u}->{v} is missing its similarity attribute"

print("✅ Validation passed:")
print(f"   GraphML round-trip matches in-memory graph: {mmG_reloaded.number_of_nodes()} nodes, {mmG_reloaded.number_of_edges()} edges")
print(f"   Node types match ontology: {sorted(found_node_types)}")
print(f"   Relationship types match ontology: {sorted(found_relations)}")
print(f"   All PREDICTED_FINDING / VISUALLY_SIMILAR_TO edges carry their required numeric attribute")

## How this graph differs from the structured knowledge graph

| | Structured KG (earlier in this notebook) | Multimedia KG (this section) |
|---|---|---|
| Source | `Data_Entry_2017_v2020.csv` metadata table | `image_predictions` + `image_embeddings`, produced directly from pixels |
| Node types | `Patient`, `Image`, `Finding`, `ViewPosition` | `Image`, `PredictedFinding`, `VisualCluster` |
| How nodes are found | Reading distinct values out of spreadsheet columns | `PredictedFinding` from the model's own output vocabulary; `VisualCluster` from k-means over embeddings — a node type with no column to read it from |
| Edge semantics | Categorical / deterministic — an image *either has* a finding label or it doesn't | Probabilistic / learned — `PREDICTED_FINDING` carries a probability, `VISUALLY_SIMILAR_TO` carries a cosine similarity, both continuous-valued |
| `HAS_FINDING` vs `PREDICTED_FINDING` | Ground-truth label a radiologist recorded | Model belief, thresholded at `PRED_THRESHOLD`, and could change with a different threshold, a different model, or fine-tuning |
| Relationships with no metadata counterpart | — none — every edge maps to a column | `VISUALLY_SIMILAR_TO` and `BELONGS_TO_CLUSTER` — these exist only because embeddings let images be compared to each other directly, something no spreadsheet column encodes |
| Patient / view context | First-class nodes, since the CSV carries that identity | Absent — this pipeline never reads patient IDs or view positions, only pixels and model outputs |
| Reproducibility | Identical every run — it's just reading a CSV | Depends on model weights, `PRED_THRESHOLD`, `K_NEAREST`, and k-means initialization — re-running with different settings changes the graph |

In short: the structured graph encodes what people already wrote down about the images. This graph
encodes what the deep learning model *saw* in them — its confidence in each disease, and which
images it considers visually alike — with every edge traceable to a specific number the network
produced, not a column someone filled in.

## Stopping point

This section stops here, having exported `chestxray14_multimedia_knowledge_graph.graphml`. No
further modeling, querying, or merging with the structured knowledge graph is performed.

# Comparing the Two Knowledge Graphs

This notebook has now built two knowledge graphs from the same underlying dataset, taking very
different routes to get there:

- **Structured KG** (`G`) — built exclusively from `Data_Entry_2017_v2020.csv` metadata columns
- **Multimedia KG** (`mmG`) — built exclusively from a pretrained deep learning model's predictions
  and embeddings, computed directly from raw pixels

The tables below compare them directly. Where a metric can be computed from the live graph objects
still in memory, it is computed programmatically rather than asserted; qualitative dimensions
(semantic richness, explainability, etc.) are discussed narratively alongside a summary table.

## Ontology comparison

| | Structured KG | Multimedia KG |
|---|---|---|
| Data source | `Data_Entry_2017_v2020.csv` (spreadsheet) | `image_predictions`, `image_embeddings` (model outputs) |
| Construction basis | Reading distinct column values | Running a forward pass through a pretrained CNN |
| Node types (count) | 4 — `Patient`, `Image`, `Finding`, `ViewPosition` | 3 — `Image`, `PredictedFinding`, `VisualCluster` |
| Relationship types (count) | 4 — `HAS_IMAGE`, `HAS_FINDING`, `TAKEN_IN_VIEW`, `CO_OCCURS_WITH` | 4 — `PREDICTED_FINDING`, `VISUALLY_SIMILAR_TO`, `BELONGS_TO_CLUSTER`, `CO_PREDICTED_WITH` |
| Edge value type | Categorical (present / absent) | Continuous (probability, cosine similarity) |
| Ground truth vs. inference | Encodes recorded ground-truth labels | Encodes model belief, which may be wrong |

## Node types and relationship types, side by side

| Node type | In Structured KG | In Multimedia KG |
|---|---|---|
| Patient | ✅ | ❌ (patient identity never enters the pixel pipeline) |
| Image | ✅ | ✅ |
| Finding (ground truth) | ✅ (`Finding`) | ❌ |
| PredictedFinding (model belief) | ❌ | ✅ |
| ViewPosition | ✅ | ❌ |
| VisualCluster | ❌ | ✅ (no metadata column corresponds to this) |

| Relationship type | In Structured KG | In Multimedia KG |
|---|---|---|
| Patient→Image ownership | ✅ (`HAS_IMAGE`) | ❌ |
| Image→Finding (categorical) | ✅ (`HAS_FINDING`) | ❌ |
| Image→Finding (probabilistic) | ❌ | ✅ (`PREDICTED_FINDING`) |
| Image→ViewPosition | ✅ (`TAKEN_IN_VIEW`) | ❌ |
| Image→Image similarity | ❌ | ✅ (`VISUALLY_SIMILAR_TO`) |
| Image→Cluster membership | ❌ | ✅ (`BELONGS_TO_CLUSTER`) |
| Finding↔Finding co-occurrence | ✅ (`CO_OCCURS_WITH`, ground truth) | ✅ (`CO_PREDICTED_WITH`, model belief) |

## Quantitative comparison — computed from the live graph objects

`G` (structured) was built over the full metadata CSV, while `mmG` (multimedia) was built over only
`N_SAMPLE_IMAGES` images from Step 2 of the multimedia pipeline — a scale difference that is itself
part of the comparison (see the note after the table), not an oversight.

**Code Explanation:** `summarize_graph()` is written once and reused for every graph comparison in this notebook (including the Precision Medicine KG comparison later), so all three graphs are measured on identical terms.

**Expected output:** `comparison_df`, one row per graph.

**Discussion:** see the note below on why `mmG`'s smaller scale affects this table's density column.

In [ ]:
# Comparison — graph-level statistics computed directly from G and mmG
def summarize_graph(Graph, name):
    node_types = Counter(nx.get_node_attributes(Graph, "node_type").values())
    edge_types = Counter(d["relation"] for _, _, d in Graph.edges(data=True))
    undirected_simple = nx.Graph(Graph.to_undirected())  # collapse multi-edges for clustering coeff
    degrees = [d for _, d in Graph.degree()]
    return {
        "Graph": name,
        "Nodes": Graph.number_of_nodes(),
        "Edges": Graph.number_of_edges(),
        "Node types": len(node_types),
        "Relationship types": len(edge_types),
        "Density": round(nx.density(Graph), 5),
        "Avg degree": round(float(np.mean(degrees)), 2) if degrees else 0.0,
        "Avg clustering coeff": round(nx.average_clustering(undirected_simple), 4),
        "Connected components": nx.number_connected_components(Graph.to_undirected()),
    }

comparison_df = pd.DataFrame([
    summarize_graph(G, "Structured KG"),
    summarize_graph(mmG, "Multimedia KG"),
]).set_index("Graph")

print("✅ Computed live graph statistics for both knowledge graphs")
comparison_df

**Reading the density comparison:** small graphs are mechanically denser than large ones (fewer
possible node pairs to fail to connect), so `mmG`'s higher density mostly reflects its much smaller
scale (built from `N_SAMPLE_IMAGES` images), not an inherently richer structure. A fair apples-to-
apples density comparison would need the multimedia pipeline run over a comparably sized image set.

## Qualitative comparison — semantic richness, complexity, explainability, scalability, reasoning

| Dimension | Structured KG | Multimedia KG |
|---|---|---|
| **Semantic richness** | High for *recorded* facts (who, which view, which label) but flat — no notion of visual similarity or model confidence | High for *learned* visual structure (similarity, clustering, confidence) but blind to anything not captured by the model, e.g. patient identity |
| **Preprocessing complexity** | Low — pandas parsing of a CSV, string splitting on `\|` | High — image decode/resize/normalize, GPU forward pass, embedding extraction, similarity computation, clustering |
| **Explainability** | High — every edge traces to one spreadsheet cell; trivial to audit | Lower — a `PREDICTED_FINDING` edge reflects a deep network's internal computation; a `VISUALLY_SIMILAR_TO` edge reflects a 1024-d embedding distance, neither of which is human-legible without extra tooling (e.g. Grad-CAM) |
| **Scalability** | Scales linearly with CSV rows; construction is effectively free | Scales with the cost of running inference on every image (GPU time) and, for `VISUALLY_SIMILAR_TO` and clustering, similarity computation that grows at least quadratically in naive form unless approximate nearest-neighbor methods are used |
| **Reasoning capability** | Strong for categorical, rule-like queries ("which images from female patients over 60 have Cardiomegaly?") | Strong for similarity-based and confidence-weighted queries ("find images visually closest to this one", "which findings does the model believe co-occur, and how confidently?") — weaker for anything needing recorded ground truth |

Neither graph is strictly better: they encode different kinds of knowledge (recorded facts vs.
learned representations) and are suited to different downstream questions.

## Extended Quantitative Comparison — Centrality, Construction Time, and Memory Usage

The comparison above covers size, density, and connectivity. Three more dimensions matter for a
production decision: how central individual nodes are (which findings/images act as hubs), how long
each pipeline takes to assemble a graph from its already-computed inputs, and how much memory the
resulting graph occupies. All three are measured directly from the live `G` and `mmG` objects and
from a lightweight, isolated re-run of just the graph-*assembly* step (not the upstream CSV parse or
model inference, which are separate costs already discussed qualitatively above).

**Centrality note:** betweenness centrality is expensive (`O(V·E)`) exactly, so for the larger `G` it
is estimated with `k`-node sampling (`nx.betweenness_centrality(..., k=...)`), a standard
approximation; `mmG` is small enough for the exact computation.

**Objective:** go beyond size/density into which nodes act as structural hubs, and how expensive each graph is to build and store.

**Implementation:** `centrality_summary()` (degree/betweenness/closeness, with sampled betweenness for the larger graph) and `time_and_memory()` (wraps standalone re-implementations of each graph's assembly step in `tracemalloc`).

**Expected output:** `performance_df` and `centrality_df`, joined into `extended_comparison_df`.

**Discussion:** the benchmark functions are intentionally side-effect-free copies — they never touch the live `G`/`mmG` objects used everywhere else in the notebook.

In [ ]:
# Benchmark: graph-assembly time and peak memory for both pipelines.
# These functions re-implement ONLY the assembly step already performed earlier in this notebook
# (Section 2 for the structured KG, Step 9 for the multimedia KG) as standalone, side-effect-free
# functions, purely so construction cost can be measured without touching the `G` / `mmG` objects
# already built and used everywhere else in the notebook.
def build_structured_graph_benchmark(df):
    Gb = nx.MultiDiGraph()
    all_findings = sorted(set(itertools.chain.from_iterable(df["finding_list"])))
    view_positions = sorted(df["view_position"].unique())
    for f in all_findings:
        Gb.add_node(f"Finding::{f}", node_type="Finding", name=f)
    for v in view_positions:
        Gb.add_node(f"View::{v}", node_type="ViewPosition", name=v)

    pair_counts_b = {}
    for row in df.itertuples(index=False):
        pid = f"Patient::{row.patient_id}"
        img_id = f"Image::{row.image_index}"
        Gb.add_node(pid, node_type="Patient")
        Gb.add_node(img_id, node_type="Image")
        Gb.add_edge(pid, img_id, key="HAS_IMAGE", relation="HAS_IMAGE")
        Gb.add_edge(img_id, f"View::{row.view_position}", key="TAKEN_IN_VIEW", relation="TAKEN_IN_VIEW")
        findings = row.finding_list
        for f in findings:
            Gb.add_edge(img_id, f"Finding::{f}", key=f"HAS_FINDING::{f}", relation="HAS_FINDING")
        real_findings = [f for f in findings if f != "No Finding"]
        for f1, f2 in itertools.combinations(sorted(real_findings), 2):
            pair_counts_b[(f1, f2)] = pair_counts_b.get((f1, f2), 0) + 1

    for (f1, f2), w in pair_counts_b.items():
        Gb.add_edge(f"Finding::{f1}", f"Finding::{f2}", key="CO_OCCURS_WITH", relation="CO_OCCURS_WITH", weight=w)
        Gb.add_edge(f"Finding::{f2}", f"Finding::{f1}", key="CO_OCCURS_WITH", relation="CO_OCCURS_WITH", weight=w)
    return Gb


def build_multimedia_graph_benchmark(image_predictions, image_embeddings, PATHOLOGIES, sample_paths,
                                      pred_threshold=0.5, k_nearest=3, n_clusters=4, random_state=42):
    Gb = nx.MultiDiGraph()
    img_ids_b = [os.path.basename(p) for p in sample_paths]
    for img_id in img_ids_b:
        Gb.add_node(f"Image::{img_id}", node_type="Image")
    for finding in PATHOLOGIES:
        Gb.add_node(f"PredictedFinding::{finding}", node_type="PredictedFinding")

    pair_counts_b = {}
    for img_id, path in zip(img_ids_b, sample_paths):
        probs = image_predictions[path]
        above = [f for f, p in probs.items() if p >= pred_threshold]
        for finding in above:
            Gb.add_edge(f"Image::{img_id}", f"PredictedFinding::{finding}",
                        key=f"PREDICTED_FINDING::{finding}", relation="PREDICTED_FINDING",
                        probability=float(probs[finding]))
        for f1, f2 in itertools.combinations(sorted(above), 2):
            pair_counts_b[(f1, f2)] = pair_counts_b.get((f1, f2), 0) + 1
    for (f1, f2), w in pair_counts_b.items():
        Gb.add_edge(f"PredictedFinding::{f1}", f"PredictedFinding::{f2}",
                    key="CO_PREDICTED_WITH", relation="CO_PREDICTED_WITH", weight=w)
        Gb.add_edge(f"PredictedFinding::{f2}", f"PredictedFinding::{f1}",
                    key="CO_PREDICTED_WITH", relation="CO_PREDICTED_WITH", weight=w)

    emb_matrix = np.stack([image_embeddings[p] for p in sample_paths])
    sim_matrix = cosine_similarity(emb_matrix)
    for i, img_id_i in enumerate(img_ids_b):
        sims = sorted([(j, sim_matrix[i, j]) for j in range(len(img_ids_b)) if j != i],
                      key=lambda t: t[1], reverse=True)
        for j, sim in sims[:k_nearest]:
            Gb.add_edge(f"Image::{img_id_i}", f"Image::{img_ids_b[j]}",
                        key="VISUALLY_SIMILAR_TO", relation="VISUALLY_SIMILAR_TO", similarity=float(sim))

    n_clust = min(n_clusters, len(sample_paths))
    kmeans_b = KMeans(n_clusters=n_clust, random_state=random_state, n_init=10)
    assigns = kmeans_b.fit_predict(emb_matrix)
    for c in range(n_clust):
        Gb.add_node(f"VisualCluster::{c}", node_type="VisualCluster")
    for img_id, c in zip(img_ids_b, assigns):
        Gb.add_edge(f"Image::{img_id}", f"VisualCluster::{c}", key="BELONGS_TO_CLUSTER", relation="BELONGS_TO_CLUSTER")
    return Gb


def time_and_memory(build_fn, *args, **kwargs):
    tracemalloc.start()
    t0 = time.time()
    result = build_fn(*args, **kwargs)
    elapsed = time.time() - t0
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    serialized_kb = sys.getsizeof(pickle.dumps(result)) / 1024
    return result, elapsed, peak / (1024 * 1024), serialized_kb


_, structured_time_s, structured_peak_mb, structured_serialized_kb = time_and_memory(
    build_structured_graph_benchmark, df
)
_, multimedia_time_s, multimedia_peak_mb, multimedia_serialized_kb = time_and_memory(
    build_multimedia_graph_benchmark, image_predictions, image_embeddings, PATHOLOGIES, sample_paths,
    pred_threshold=PRED_THRESHOLD, k_nearest=K_NEAREST, n_clusters=N_CLUSTERS, random_state=RANDOM_STATE,
)

performance_df = pd.DataFrame([
    {"Graph": "Structured KG", "Assembly time (s)": round(structured_time_s, 4),
     "Peak memory during assembly (MB)": round(structured_peak_mb, 3),
     "Serialized size (KB)": round(structured_serialized_kb, 1)},
    {"Graph": "Multimedia KG", "Assembly time (s)": round(multimedia_time_s, 4),
     "Peak memory during assembly (MB)": round(multimedia_peak_mb, 3),
     "Serialized size (KB)": round(multimedia_serialized_kb, 1)},
]).set_index("Graph")

print("✅ Benchmarked graph-assembly time and memory for both pipelines")
performance_df

In [ ]:
# Centrality measures for both graphs (degree, approximate betweenness, closeness)
def centrality_summary(Graph, name, betweenness_sample=300, seed=42):
    UG = nx.Graph(Graph.to_undirected())  # simple undirected view for standard centrality definitions
    n = UG.number_of_nodes()

    deg_cent = nx.degree_centrality(UG)
    k = None if n <= betweenness_sample else betweenness_sample
    between_cent = nx.betweenness_centrality(UG, k=k, seed=seed, normalized=True)

    largest_cc_nodes = max(nx.connected_components(UG), key=len)
    CCG = UG.subgraph(largest_cc_nodes)
    close_cent = nx.closeness_centrality(CCG)

    top_degree = max(deg_cent, key=deg_cent.get)
    top_between = max(between_cent, key=between_cent.get)
    top_close = max(close_cent, key=close_cent.get)

    return {
        "Graph": name,
        "Mean degree centrality": round(float(np.mean(list(deg_cent.values()))), 5),
        "Top hub node (degree)": top_degree,
        "Mean betweenness centrality (approx)": round(float(np.mean(list(between_cent.values()))), 5),
        "Top bridge node (betweenness)": top_between,
        "Mean closeness centrality (largest CC)": round(float(np.mean(list(close_cent.values()))), 5),
        "Top reachable node (closeness)": top_close,
    }

centrality_df = pd.DataFrame([
    centrality_summary(G, "Structured KG", betweenness_sample=300),
    centrality_summary(mmG, "Multimedia KG", betweenness_sample=300),
]).set_index("Graph")

print("✅ Computed centrality measures (degree, approx. betweenness, closeness) for both graphs")
centrality_df

In [ ]:
# Combined table: topology + performance + centrality, all on one row per graph
extended_comparison_df = comparison_df.join(performance_df).join(centrality_df)
print("✅ Combined comparison table (topology + performance + centrality)")
extended_comparison_df

## Comparison Visualizations

**Objective:** turn the numeric comparison tables above into an at-a-glance visual.

**Implementation:** grouped bar charts (size/ontology, performance, centrality) and a 1-5 radar chart for the qualitative dimensions discussed above.

**Expected output:** two saved PNGs.

**Discussion:** the radar chart's scores are a judgment call, not a computed metric — flagged explicitly in the code comment above it.

In [ ]:
# Grouped bar charts: size/ontology breadth, construction time & memory, mean centrality
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

size_metrics = ["Nodes", "Edges", "Node types", "Relationship types"]
extended_comparison_df[size_metrics].plot(kind="bar", ax=axes[0], rot=0, logy=True)
axes[0].set_title("Graph size & ontology breadth (log scale)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=7)

perf_metrics = ["Assembly time (s)", "Peak memory during assembly (MB)"]
extended_comparison_df[perf_metrics].plot(kind="bar", ax=axes[1], rot=0, color=["#4C72B0", "#C44E52"])
axes[1].set_title("Construction time & memory")

cent_metrics = ["Mean degree centrality", "Mean betweenness centrality (approx)", "Mean closeness centrality (largest CC)"]
extended_comparison_df[cent_metrics].plot(kind="bar", ax=axes[2], rot=0)
axes[2].set_title("Mean centrality measures")
axes[2].legend(fontsize=6)

plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "compare_01_quantitative_bars.png"), dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# Radar chart: qualitative dimensions, scored 1-5 for a visual read of the table above.
# Scores are the authors' judgment call, consistent with the qualitative discussion in this notebook;
# note the axes are NOT all "higher = better" in the same sense (e.g. high preprocessing complexity
# is a cost, not a benefit) — read each axis on its own terms, not as an aggregate score.
qualitative_scores = pd.DataFrame({
    "Structured KG": [2, 4, 5, 2, 5, 4, 3],
    "Multimedia KG": [4, 2, 2, 4, 4, 3, 4],
}, index=[
    "Ontology complexity", "Semantic richness (recorded facts)", "Explainability",
    "Preprocessing complexity", "Scalability", "Reasoning capability (categorical)",
    "Downstream AI usability",
])

categories = list(qualitative_scores.index)
n_cat = len(categories)
angles = [n / float(n_cat) * 2 * np.pi for n in range(n_cat)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(7.5, 7.5), subplot_kw=dict(polar=True))
for col, color in zip(qualitative_scores.columns, ["#4C72B0", "#C44E52"]):
    values = qualitative_scores[col].tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, label=col, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=8)
ax.set_ylim(0, 5)
ax.set_title("Qualitative dimensions — Structured KG vs. Multimedia KG (1=low, 5=high)", fontsize=11, y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "compare_02_qualitative_radar.png"), dpi=180, bbox_inches="tight")
plt.show()

# A Simple Disease-Specific Precision Medicine Knowledge Graph

The two graphs above are both organized around *images*. This section builds a third, much smaller
graph organized around the **14 ChestX-ray14 disease labels themselves**, enriched with simplified
clinical context — the kind of graph a precision-medicine or clinical-decision-support tool might
start from. It reuses the finding co-occurrence counts already computed during the structured KG
build (`finding_pair_counts.pkl`), so comorbidity edges are grounded in this dataset's real
statistics rather than invented.

**This is an illustrative, simplified graph for demonstration purposes — not a validated clinical
knowledge base.** The risk factor / treatment / biomarker associations below are common textbook
associations included to show the pattern of a precision-medicine ontology, not a substitute for
a curated clinical resource or medical advice.

## Ontology for the Precision Medicine Knowledge Graph

**Node types (5):**
1. `Disease` — id: one of the 14 ChestX-ray14 finding names
2. `RiskFactor` — id: risk factor name (e.g. Smoking, Age > 60)
3. `Treatment` — id: typical management/treatment name
4. `Biomarker` — id: a lab or imaging marker commonly associated with the disease
5. `BodySystem` — id: anatomical/physiological system affected

**Relationship types (5):**
1. `(Disease) -[:HAS_RISK_FACTOR]-> (RiskFactor)`
2. `(Disease) -[:TREATED_WITH]-> (Treatment)`
3. `(Disease) -[:INDICATED_BY]-> (Biomarker)`
4. `(Disease) -[:AFFECTS_SYSTEM]-> (BodySystem)`
5. `(Disease) -[:COMORBID_WITH]-> (Disease)` — derived, symmetric, attr `weight`, reused directly
   from the structured KG's `finding_pair_counts` (this dataset's real co-occurrence statistics)

**Objective:** design a small, disease-centric ontology suited to point-of-care lookups.

**Implementation:** see the ontology below — 5 node types, 5 relationship types, with `COMORBID_WITH` reused directly from the structured graph's real co-occurrence counts.

**Expected output:** none (design-only markdown); realized by PM1-PM3 below.

**Discussion:** grounding `COMORBID_WITH` in this dataset's actual statistics (rather than inventing comorbidity weights) keeps that one relationship type honest, even though the risk-factor/treatment/biomarker content is illustrative.

In [ ]:
# PM1 — Simplified clinical knowledge per disease (illustrative, not a validated clinical KB)
DISEASE_KNOWLEDGE = {
    "Atelectasis": {
        "risk_factors": ["Post-surgical immobility", "Mucus plugging"],
        "treatments": ["Chest physiotherapy", "Bronchoscopy"],
        "biomarkers": ["Reduced breath sounds"],
        "body_system": "Respiratory",
    },
    "Cardiomegaly": {
        "risk_factors": ["Hypertension", "Chronic heart failure"],
        "treatments": ["Diuretics", "ACE inhibitors"],
        "biomarkers": ["BNP / NT-proBNP"],
        "body_system": "Cardiovascular",
    },
    "Effusion": {
        "risk_factors": ["Heart failure", "Malignancy"],
        "treatments": ["Thoracentesis", "Diuretics"],
        "biomarkers": ["Pleural fluid protein/LDH ratio"],
        "body_system": "Respiratory",
    },
    "Infiltration": {
        "risk_factors": ["Infection", "Aspiration"],
        "treatments": ["Antibiotics", "Supportive care"],
        "biomarkers": ["CRP", "WBC count"],
        "body_system": "Respiratory",
    },
    "Mass": {
        "risk_factors": ["Smoking history", "Age > 50"],
        "treatments": ["Biopsy", "Oncology referral"],
        "biomarkers": ["Tumor markers (case-dependent)"],
        "body_system": "Respiratory",
    },
    "Nodule": {
        "risk_factors": ["Smoking history", "Prior granulomatous disease"],
        "treatments": ["Serial imaging follow-up", "Biopsy if high-risk"],
        "biomarkers": ["Nodule growth rate on CT"],
        "body_system": "Respiratory",
    },
    "Pneumonia": {
        "risk_factors": ["Immunosuppression", "Age extremes"],
        "treatments": ["Antibiotics", "Oxygen therapy"],
        "biomarkers": ["CRP", "Procalcitonin"],
        "body_system": "Respiratory",
    },
    "Pneumothorax": {
        "risk_factors": ["Tall thin build", "Trauma", "COPD"],
        "treatments": ["Chest tube", "Observation (small cases)"],
        "biomarkers": ["Reduced breath sounds", "Hyperresonance"],
        "body_system": "Respiratory",
    },
    "Consolidation": {
        "risk_factors": ["Infection", "Aspiration"],
        "treatments": ["Antibiotics", "Supportive care"],
        "biomarkers": ["CRP", "WBC count"],
        "body_system": "Respiratory",
    },
    "Edema": {
        "risk_factors": ["Heart failure", "Renal failure"],
        "treatments": ["Diuretics", "Fluid restriction"],
        "biomarkers": ["BNP / NT-proBNP"],
        "body_system": "Cardiovascular",
    },
    "Emphysema": {
        "risk_factors": ["Smoking history", "Alpha-1 antitrypsin deficiency"],
        "treatments": ["Bronchodilators", "Pulmonary rehab", "Smoking cessation"],
        "biomarkers": ["Reduced FEV1/FVC ratio"],
        "body_system": "Respiratory",
    },
    "Fibrosis": {
        "risk_factors": ["Occupational exposure", "Chronic autoimmune disease"],
        "treatments": ["Antifibrotic agents", "Pulmonary rehab"],
        "biomarkers": ["Restrictive pattern on spirometry"],
        "body_system": "Respiratory",
    },
    "Pleural_Thickening": {
        "risk_factors": ["Asbestos exposure", "Prior pleural infection"],
        "treatments": ["Monitoring", "Decortication (severe cases)"],
        "biomarkers": ["Restrictive pattern on spirometry"],
        "body_system": "Respiratory",
    },
    "Hernia": {
        "risk_factors": ["Obesity", "Chronic increased intra-abdominal pressure"],
        "treatments": ["Surgical repair", "Lifestyle modification"],
        "biomarkers": ["Symptom-based (reflux, chest discomfort)"],
        "body_system": "Gastrointestinal",
    },
}

print(f"✅ Simplified clinical knowledge defined for {len(DISEASE_KNOWLEDGE)} of the 14 disease labels")

In [ ]:
# PM2 — Build the Precision Medicine Knowledge Graph with NetworkX
pmG = nx.MultiDiGraph(name="ChestXray14_PrecisionMedicineKnowledgeGraph")

for disease, info in DISEASE_KNOWLEDGE.items():
    pmG.add_node(f"Disease::{disease}", node_type="Disease", name=disease)

    for rf in info["risk_factors"]:
        pmG.add_node(f"RiskFactor::{rf}", node_type="RiskFactor", name=rf)
        pmG.add_edge(f"Disease::{disease}", f"RiskFactor::{rf}",
                     key=f"HAS_RISK_FACTOR::{rf}", relation="HAS_RISK_FACTOR")

    for tx in info["treatments"]:
        pmG.add_node(f"Treatment::{tx}", node_type="Treatment", name=tx)
        pmG.add_edge(f"Disease::{disease}", f"Treatment::{tx}",
                     key=f"TREATED_WITH::{tx}", relation="TREATED_WITH")

    for bm in info["biomarkers"]:
        pmG.add_node(f"Biomarker::{bm}", node_type="Biomarker", name=bm)
        pmG.add_edge(f"Disease::{disease}", f"Biomarker::{bm}",
                     key=f"INDICATED_BY::{bm}", relation="INDICATED_BY")

    body_system = info["body_system"]
    pmG.add_node(f"BodySystem::{body_system}", node_type="BodySystem", name=body_system)
    pmG.add_edge(f"Disease::{disease}", f"BodySystem::{body_system}",
                 key="AFFECTS_SYSTEM", relation="AFFECTS_SYSTEM")

# --- COMORBID_WITH, reused directly from the structured KG's real co-occurrence counts ---
with open(os.path.join(KG_DIR, "finding_pair_counts.pkl"), "rb") as fh:
    real_pair_counts = pickle.load(fh)

n_comorbid = 0
for (f1, f2), w in real_pair_counts.items():
    if f1 in DISEASE_KNOWLEDGE and f2 in DISEASE_KNOWLEDGE:
        pmG.add_edge(f"Disease::{f1}", f"Disease::{f2}",
                     key="COMORBID_WITH", relation="COMORBID_WITH", weight=w)
        pmG.add_edge(f"Disease::{f2}", f"Disease::{f1}",
                     key="COMORBID_WITH", relation="COMORBID_WITH", weight=w)
        n_comorbid += 2

print(f"✅ Precision Medicine KG built: {pmG.number_of_nodes()} nodes, {pmG.number_of_edges()} edges")
print(f"   ({n_comorbid} COMORBID_WITH edges reused from this dataset's real co-occurrence counts)")

## Visualize the Precision Medicine Knowledge Graph

**Objective:** confirm `pmG`'s structure visually and compute its solo statistics.

**Implementation:** the same `COLORS`-keyed plotting pattern used for `G` and `mmG`, plus `summarize_graph()` reused a third time.

**Expected output:** one plot plus `pm_stats_df`.

**Discussion:** reusing `summarize_graph()` here (rather than writing new statistics code) is what makes the three-way comparison in the next section apples-to-apples.

In [ ]:
# PM3 — Visualize the Precision Medicine Knowledge Graph
COLORS_PM = {
    "Disease": "#C44E52",
    "RiskFactor": "#DD8452",
    "Treatment": "#55A868",
    "Biomarker": "#8172B2",
    "BodySystem": "#4C72B0",
}
size_map_pm = {"Disease": 550, "RiskFactor": 260, "Treatment": 260, "Biomarker": 260, "BodySystem": 420}

fig, ax = plt.subplots(figsize=(15, 12))
pos_pm = nx.spring_layout(pmG, seed=RANDOM_STATE, k=0.5, iterations=80)

for nt, color in COLORS_PM.items():
    nodelist = [n for n, d in pmG.nodes(data=True) if d.get("node_type") == nt]
    nx.draw_networkx_nodes(pmG, pos_pm, nodelist=nodelist, node_color=color,
                            node_size=size_map_pm[nt], alpha=0.9, ax=ax,
                            edgecolors="white", linewidths=1)

edge_colors_pm = {
    "HAS_RISK_FACTOR": "#DD8452",
    "TREATED_WITH": "#55A868",
    "INDICATED_BY": "#8172B2",
    "AFFECTS_SYSTEM": "#4C72B0",
    "COMORBID_WITH": "#999999",
}
for rel, color in edge_colors_pm.items():
    elist = [(u, v) for u, v, d in pmG.edges(data=True) if d.get("relation") == rel]
    nx.draw_networkx_edges(pmG, pos_pm, edgelist=elist, edge_color=color, alpha=0.45,
                            arrows=True, arrowsize=7, width=1.0, ax=ax)

labels_pm = {n: d["name"] for n, d in pmG.nodes(data=True)}
nx.draw_networkx_labels(pmG, pos_pm, labels=labels_pm, font_size=7, ax=ax)

legend_elems = [Line2D([0], [0], marker="o", color="w", label=nt,
                        markerfacecolor=c, markersize=12) for nt, c in COLORS_PM.items()]
ax.legend(handles=legend_elems, loc="upper left", fontsize=10, title="Node type")
ax.set_title("Precision Medicine Knowledge Graph — ChestX-ray14 disease labels", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "pm_01_full_graph.png"), dpi=180, bbox_inches="tight")
plt.show()

pm_stats_df = pd.DataFrame([summarize_graph(pmG, "Precision Medicine KG")]).set_index("Graph")
pm_stats_df

## Precision Medicine KG vs. Structured KG (this notebook) — Quantitative Comparison

The comparisons above measured the Structured KG (`G`) against the Multimedia KG (`mmG`). This
section adds a third comparison — the Precision Medicine KG (`pmG`) just built, against the
Structured KG already in memory — using the same `summarize_graph` and `centrality_summary` helpers
defined earlier, so all three graphs in this notebook are measured on identical terms.

**Objective:** measure `pmG` against `G` on identical terms.

**Implementation:** `summarize_graph()` and `centrality_summary()`, the same helpers used for the structured-vs-multimedia comparison above.

**Expected output:** `pm_vs_structured_full_df`.

**Discussion:** see the qualitative table below for what these numbers mean in practice.

In [ ]:
# Quantitative comparison: Precision Medicine KG vs. Structured KG
pm_vs_structured_df = pd.DataFrame([
    summarize_graph(G, "Structured KG"),
    summarize_graph(pmG, "Precision Medicine KG"),
]).set_index("Graph")

pm_centrality_df = pd.DataFrame([
    centrality_summary(G, "Structured KG", betweenness_sample=300),
    centrality_summary(pmG, "Precision Medicine KG", betweenness_sample=300),
]).set_index("Graph")

pm_vs_structured_full_df = pm_vs_structured_df.join(pm_centrality_df)
print("✅ Precision Medicine KG vs. Structured KG — quantitative comparison")
pm_vs_structured_full_df

## Precision Medicine KG vs. Structured KG — Ontology Design, Structure, and Clinical Utility

| Dimension | Structured KG (`G`) | Precision Medicine KG (`pmG`) |
|---|---|---|
| **Ontology design** | Image-centric: `Patient`, `Image`, `Finding`, `ViewPosition` — organized around *acquisition events* | Disease-centric: `Disease`, `RiskFactor`, `Treatment`, `Biomarker`, `BodySystem` — organized around *clinical entities*, one hop from every disease |
| **Graph structure** | Large, sparse, star-like around each Image/Patient; `Finding` nodes (especially "No Finding") act as high-degree hubs | Small, dense, hub-and-spoke around each `Disease` node; every disease sits at the center of its own compact clinical neighborhood |
| **Reasoning capability** | Strong for *population-level, categorical* queries — "how many female patients over 60 have Cardiomegaly" | Strong for *bounded clinical* queries — "what treats this disease, and what commonly co-occurs with it" |
| **Diagnosis support** | Indirect — supports epidemiological pattern-finding (comorbidity via `CO_OCCURS_WITH`) but has no notion of risk factors, treatment, or biomarkers | Direct — a `Disease` node connects straight to its risk factors and biomarkers, closer to how a differential-diagnosis workflow reasons |
| **Disease representation** | A `Finding` node is just a label with a name — no clinical content attached | A `Disease` node is enriched with risk factors, treatments, biomarkers, and body system — a genuine (if simplified) clinical profile |
| **Personalization** | `Patient` nodes carry age/sex, so patient-level personalization is *possible* in principle, but nothing currently links a patient to disease-level clinical knowledge (no `Patient → RiskFactor` edge exists) | No `Patient` nodes at all — personalizing `pmG` would require joining back to `G`'s `Patient`/`Image` nodes (e.g. matching a patient's recorded risk factors against `HAS_RISK_FACTOR` edges), which the current ontology does not yet do |
| **Treatment recommendation potential** | None — `G` has no `Treatment` node type | Direct one-hop lookup, `Disease -[:TREATED_WITH]-> Treatment` — though it is a static, textbook-level mapping, not personalized to a specific patient's history, contraindications, or severity |

**Takeaway:** the two graphs are complementary, not competing. `G` knows *who has what recorded
finding*; `pmG` knows *what to generally do about a finding*. A genuinely personalized,
treatment-aware system would need to merge them — follow `Patient → Image → Finding` in `G`, then
continue `Finding → Disease → {RiskFactor, Treatment, Biomarker}` in `pmG` — which today requires a
manual name-matching join (`Finding.name == Disease.name`) since the two graphs are not merged into
one.

In [ ]:
# Visual comparison: Precision Medicine KG vs. Structured KG
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

pm_vs_structured_full_df[["Nodes", "Edges"]].plot(kind="bar", ax=axes[0], rot=0, logy=True)
axes[0].set_title("Nodes & edges (log scale)")

pm_vs_structured_full_df[["Density"]].plot(kind="bar", ax=axes[1], rot=0, legend=False, color="#DD8452")
axes[1].set_title("Graph density")

pm_vs_structured_full_df[["Avg degree", "Avg clustering coeff"]].plot(kind="bar", ax=axes[2], rot=0)
axes[2].set_title("Degree & clustering")

plt.tight_layout()
plt.savefig(os.path.join(KG_DIR, "compare_03_pm_vs_structured.png"), dpi=180, bbox_inches="tight")
plt.show()

## Precision Medicine KG vs. a general biomedical Knowledge Graph

General biomedical knowledge graphs (e.g. Hetionet, DRKG, SPOKE-style graphs) integrate many
databases — genes, proteins, pathways, drugs, side effects, anatomy, diseases, phenotypes — into one
very large, heterogeneous graph, usually with tens of node types and hundreds of thousands to
millions of nodes.

| | Precision Medicine KG (this section) | General biomedical KG (e.g. Hetionet/DRKG-style) |
|---|---|---|
| Scope | Narrow — 14 chest-imaging disease labels only | Broad — genes, proteins, pathways, drugs, diseases, anatomy, phenotypes across many domains |
| Node types | 5, hand-picked for this task | Dozens, spanning molecular biology to clinical entities |
| Data sources | This dataset's co-occurrence stats + a small hardcoded clinical summary | Curated integration of many structured databases and literature-mined relations |
| Relationship richness | Shallow — 5 relation types, mostly one hop from Disease | Deep — long multi-hop paths (drug → target → pathway → disease → phenotype) |
| Maintenance burden | Low — small, hand-authored, easy to update | High — requires ongoing curation/integration pipelines across many source databases |
| Typical use case | Fast, explainable lookups for a bounded diagnostic domain | Drug repurposing, mechanism discovery, hypothesis generation across biology |
| Reasoning depth | Shallow (1–2 hops) but fully traceable | Deep, multi-hop, but harder to audit any single inferred path |
| Scalability to new diseases | Manual — add a dictionary entry | Structural — new entities/relations slot into the existing schema, but validation is nontrivial |

## Which approach best supports disease diagnosis and clinical decision support?

Pulling together both comparisons in this notebook:

- **Structured KG vs. Multimedia KG:** the structured graph is the more *trustworthy* substrate for
  decision support because every edge is a recorded fact, fully auditable — appropriate where a
  clinician needs to see exactly why the system asserts something. The multimedia graph adds a
  complementary signal structured data cannot provide — visual similarity and model confidence —
  which is valuable for tasks like case retrieval ("find prior scans that look like this one") or
  triage prioritization, but it inherits the pretrained model's blind spots and is harder to audit
  edge-by-edge.

- **Precision Medicine KG vs. general biomedical KG:** the narrow, disease-specific graph built here
  is better suited to **point-of-care diagnostic support** — it answers a bounded set of questions
  (risk factors, typical treatment, comorbidity) quickly and transparently. A general biomedical KG
  is better suited to **research and mechanism discovery** — asking why a drug might work, or which
  unexplored pathway connects two conditions — where breadth and multi-hop reasoning matter more
  than every single edge being immediately explainable.

**Overall recommendation for clinical decision support at the point of care:** favor the narrower,
more explainable graphs (structured metadata KG + precision medicine KG), and treat the multimedia
KG's outputs as an auxiliary signal — surfaced to a clinician alongside its confidence score, not
substituted for a ground-truth-backed explanation. Save the general biomedical KG's breadth for
research workflows upstream of the clinic, where speculative multi-hop reasoning is the point.

## Conclusion — Key findings and recommendations

**Key findings**

1. The same ChestX-ray14 dataset supports genuinely different knowledge graphs depending on whether
   the input is a spreadsheet of recorded facts or the pixel-level output of a pretrained deep
   learning model — different node types, different relationship types, and fundamentally different
   edge semantics (categorical vs. probabilistic).
2. The structured KG is easier to build, cheaper to scale, and fully explainable; the multimedia KG
   is more expensive to build, harder to audit, but captures learned visual structure — similarity
   and confidence — that no spreadsheet column encodes.
3. A small, disease-specific precision medicine graph, grounded partly in this dataset's own
   co-occurrence statistics, offers fast and transparent diagnostic-support reasoning at the cost of
   scope; a general biomedical KG trades that transparency and simplicity for much broader,
   multi-hop reasoning power.

**Recommendations**

- Use the **structured metadata KG** as the system of record for anything that must be explainable
  or audited (e.g. why a case was flagged).
- Use the **multimedia KG** as a *complementary retrieval/triage layer*, always surfaced with its
  underlying probabilities so a clinician can judge model confidence rather than treat predictions
  as fact.
- Use a **precision medicine KG** like the one built here for fast, bounded diagnostic-support
  queries, and treat it as a starting scaffold to be validated against real clinical knowledge
  bases before any deployment.
- Reserve **general biomedical KGs** for research and hypothesis-generation workflows upstream of
  clinical use, not for direct point-of-care decisions.
- Any of these graphs intended for real clinical use requires validation by qualified clinicians and
  is out of scope for what is demonstrated in this notebook.

## Exporting All Graphs, Comparison Tables, and Visualizations

Everything produced in the comparison and precision-medicine sections is written to organized,
purpose-specific folders under `/mnt/user-data/outputs/` so it can be downloaded as one coherent
deliverable set, alongside the GraphML files already exported earlier in this notebook.

**Objective:** package every graph, table, and figure this notebook produced into one downloadable deliverable set.

**Implementation:** relocate the two GraphML files already exported, export the Precision Medicine KG's GraphML, write every comparison table to CSV, and copy every PNG generated so far — organized under `OUTPUT_DIR/{knowledge_graphs,comparison_tables,visualizations}`.

**Expected output:** a per-folder file count, printed at the end.

**Discussion:** this is the only cell in the notebook that reorganizes files rather than computing something new — everything it moves or copies was already produced above.

In [ ]:
OUT_ROOT = OUTPUT_DIR
KG_EXPORT_DIR = os.path.join(OUT_ROOT, "knowledge_graphs")  # final GraphML deliverables
TABLES_DIR = os.path.join(OUT_ROOT, "comparison_tables")
VIZ_DIR = os.path.join(OUT_ROOT, "visualizations")
for d in (KG_EXPORT_DIR, TABLES_DIR, VIZ_DIR):
    os.makedirs(d, exist_ok=True)

# --- Knowledge graphs: relocate the two already exported, add the Precision Medicine KG ---
for existing_name in ["chestxray14_knowledge_graph.graphml", "chestxray14_multimedia_knowledge_graph.graphml"]:
    src = os.path.join(OUT_ROOT, existing_name)
    if os.path.exists(src):
        shutil.move(src, os.path.join(KG_EXPORT_DIR, existing_name))

pmG_export = pmG.copy()
for n, data in pmG_export.nodes(data=True):
    for k, v in list(data.items()):
        if isinstance(v, (bool, np.bool_)):
            data[k] = str(v)
for u, v, data in pmG_export.edges(data=True):
    for k, val in list(data.items()):
        if isinstance(val, (bool, np.bool_)):
            data[k] = str(val)
pm_graphml_path = os.path.join(KG_EXPORT_DIR, "chestxray14_precision_medicine_knowledge_graph.graphml")
nx.write_graphml(pmG_export, pm_graphml_path)

# --- Comparison tables ---
comparison_df.to_csv(os.path.join(TABLES_DIR, "structured_vs_multimedia_basic_stats.csv"))
extended_comparison_df.to_csv(os.path.join(TABLES_DIR, "structured_vs_multimedia_extended_stats.csv"))
pm_vs_structured_full_df.to_csv(os.path.join(TABLES_DIR, "precision_medicine_vs_structured_stats.csv"))
pm_stats_df.to_csv(os.path.join(TABLES_DIR, "precision_medicine_kg_solo_stats.csv"))

# --- Visualizations: copy every PNG generated so far in this session ---
# (KG_DIR, not KG_EXPORT_DIR, is where every plt.savefig() call in this notebook wrote its PNG)
png_files = sorted(glob.glob(os.path.join(KG_DIR, "*.png")))
for f in png_files:
    shutil.copy(f, os.path.join(VIZ_DIR, os.path.basename(f)))

print("✅ Export complete")
print(f"   Knowledge graphs  -> {KG_EXPORT_DIR}  ({len(os.listdir(KG_EXPORT_DIR))} files)")
print(f"   Comparison tables -> {TABLES_DIR}  ({len(os.listdir(TABLES_DIR))} files)")
print(f"   Visualizations    -> {VIZ_DIR}  ({len(os.listdir(VIZ_DIR))} files)")

## Final Validation Summary

**Objective:** confirm this notebook is safe to run top-to-bottom in a fresh Google Colab session.

**Implementation:** every code cell was parsed with Python's `ast` module (syntax check), the notebook JSON was validated against the `nbformat` v4.5 schema, and every hardcoded, environment-specific path was replaced with a variable defined once in **Global Setup** (`BASE_DIR`, `DATA_DIR`, `KG_DIR`, `MODELS_DIR`, `OUTPUT_DIR`, `CSV_PATH`, `CSV_INPUT_PATH`, `IMAGE_DIR`, `METADATA_CSV`) so no cell below Global Setup contains a hardcoded filesystem path.

**Expected output:** in Colab, the first three cells (dependency install, Global Setup imports, path configuration) run without edits. Every later cell then runs unmodified **provided the two input sources below are placed under `DATA_DIR`** (or `IMAGE_DIR`/`CSV_PATH`/`CSV_INPUT_PATH` are edited in Global Setup to point elsewhere, e.g. a mounted Google Drive):

1. `Data_Entry_2017_v2020.csv` — NIH metadata table
2. A folder of raw ChestX-ray14 `.png`/`.jpg` files, at `IMAGE_DIR`

`image_features_dataset.csv` no longer needs to be supplied separately — Sections 11-12 generate it from inputs (1) and (2) above the first time the notebook is run.

**Discussion:** what could **not** be verified from this environment is a live, networked, GPU-backed run — this sandbox has no internet access, no GPU, and none of the real ChestX-ray14 files, so package installation, pretrained-weight downloads, and the file-existence `assert`s throughout this notebook cannot execute here. Every code cell that *doesn't* depend on those external inputs — both knowledge-graph builds' internal logic, the classical-ML pipeline's logic, and every visualization/comparison function — was additionally exercised end-to-end against synthetic stand-in data of the same shape as the real dataset, which caught and fixed several bugs (hardcoded sandbox paths, a dead code block, a variable-name collision between two unrelated `KG_DIR` uses) now corrected above. Run the dependency-install cell and this notebook in Colab, with the metadata CSV and raw images in place, for a genuine top-to-bottom execution confirmation.